In [1]:
%cd /app

/app


In [2]:
import argparse
import os
import sys

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import torch
torch.multiprocessing.set_start_method('spawn')

import jax
from lob.encoding import Vocab, Message_Tokenizer

from lob import inference_no_errcorr as inference
from lob.init_train import init_train_state, load_checkpoint, load_metadata, load_args_from_checkpoint

from lob import inference_no_errcorr as inference
import lob.encoding as encoding
import preproc as preproc

import jax.numpy as jnp
import numpy as np

from pathlib import Path
import os

import pandas as pd

import pandas as pd
import plotly.graph_objs as go
import yaml

from filtration_utils import summary_table, build_zero_padded_series, plot_midprice_series_with_insertions, prepare_volatility_filtered_series, plot_midprice_series_with_mean_std

2025-09-03 10:14:35.401123: W external/xla/xla/service/gpu/nvptx_compiler.cc:718] The NVIDIA driver's CUDA version is 12.8 which is older than the ptxas CUDA version (12.9.41). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-09-03 10:14:37.702568: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
experiment_name = 'exp_24_20250805_101039_gen_buy_whole_55'  #       exp_85_20250823_020427_1024


# Load the sample day map CSV file
sample_day_map = pd.read_csv('/app/sample_day_map_1024.csv')
print(sample_day_map.head())



CONFIG_PATH = f"/app/data_saved/{experiment_name}/used_config.yaml"

# Load YAML config
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

# Extract values
num_insertions      = config["num_insertions"]
num_coolings        = config["num_coolings"]
midprice_step_size  = config["midprice_step_size"]
hist_msgs           = config["n_messages"]
n_gen_msgs          = config["n_gen_msgs"]
Direction           = config["DIRECTION_i"]

print(f"Aggressive {'buy' if Direction == 0 else 'sell'}\n")
print(f'num_insertions: {num_insertions}')
print(f'num_coolings: {num_coolings}')
print(f'midprice_step_size: {midprice_step_size}')
print(f'hist_msgs: {hist_msgs}')
print(f'n_gen_msgs: {n_gen_msgs}')

   sample_id                                         file_name  highest_price  \
0          6  GOOG_2023-01-06_34200000_57600000_message_10.csv         885700   
1          7  GOOG_2023-01-06_34200000_57600000_message_10.csv         885700   
2          9  GOOG_2023-01-06_34200000_57600000_message_10.csv         885700   
3         15  GOOG_2023-01-06_34200000_57600000_message_10.csv         885700   
4         17  GOOG_2023-01-06_34200000_57600000_message_10.csv         885700   

   lowest_price  execution_sum  
0        854800        3539292  
1        854800        3539292  
2        854800        3539292  
3        854800        3539292  
4        854800        3539292  
Aggressive buy

num_insertions: 20
num_coolings: 30
midprice_step_size: 1
hist_msgs: 500
n_gen_msgs: 50


In [4]:
order_volume = config["order_volume"]
order_volume_ratio = config["order_volume_ratio"]
use_relative_volume = config["use_relative_volume"]

print(f'use_relative_volume: {use_relative_volume}')
print(f'order_volume: {order_volume}')
print(f'order_volume_ratio: {order_volume_ratio}')

# use_sample_file = config["use_sample_file"]
# sample_file_path = config["sample_file_path"]
# start_batch = config["start_batch"]
# end_batch = config["end_batch"]

# print(f'\nuse_sample_file: {use_sample_file}')
# print(f'sample_file_path: {sample_file_path}')
# print(f'start_batch: {start_batch}')
# print(f'end_batch: {end_batch}')

use_relative_volume: True
order_volume: 75
order_volume_ratio: 1.0


In [5]:
hist_steps = hist_msgs // midprice_step_size       # 500
gen_steps = n_gen_msgs // midprice_step_size     # 50
gen_block = gen_steps + 1                        # 51

merged = summary_table(experiment_name)
x, all_series = build_zero_padded_series(hist_msgs, n_gen_msgs, midprice_step_size, merged)

print(merged)

       id                                        merged_data
0     114  [904100, 904100, 904100, 904100, 904100, 90410...
1     648  [909800, 909800, 909800, 909800, 909800, 90980...
2    1895  [891900, 891900, 891900, 891900, 891900, 89190...
3    2486  [895200, 895200, 895200, 895200, 895200, 89520...
4    2787  [897700, 897700, 897700, 897700, 897700, 89770...
5    3489  [897200, 897200, 897200, 897200, 897200, 89720...
6    4447  [886800, 886800, 886800, 886800, 886800, 88680...
7    4668  [887000, 887000, 887000, 887000, 887000, 88700...
8    6556  [893500, 893500, 893500, 893500, 893500, 89350...
9    6779  [887500, 887500, 887500, 887500, 887500, 88750...
10   8306  [884000, 884000, 884000, 884000, 884000, 88400...
11   8349  [883700, 883700, 883700, 883700, 883700, 88370...
12   8604  [885900, 885900, 885900, 885900, 885900, 88590...
13   8858  [884800, 884800, 884800, 884700, 884700, 88480...
14  10954  [871700, 871700, 871700, 871700, 871700, 87170...
15  11171  [869800, 8698

In [6]:
fig = plot_midprice_series_with_insertions(
    merged,
    all_series,
    x,
    hist_steps,
    gen_block,
    num_insertions,
    num_coolings,
    n_gen_msgs,
    midprice_step_size
)
fig.show()

insertion positions: [551, 602, 653, 704, 755, 806, 857, 908, 959, 1010, 1061, 1112, 1163, 1214, 1265, 1316, 1367, 1418, 1469, 1520]
cooling positions:   [1570, 1620, 1670, 1720, 1770, 1820, 1870, 1920, 1970, 2020, 2070, 2120, 2170, 2220, 2270, 2320, 2370, 2420, 2470, 2520, 2570, 2620, 2670, 2720, 2770, 2820, 2870, 2920, 2970, 3020]


In [7]:
# x, all_series, merged, hist_steps, gen_block = prepare_volatility_filtered_series(merged, hist_msgs, n_gen_msgs, midprice_step_size, volatility_cutoff=0.1)

In [8]:
fig, mean_series, std_series = plot_midprice_series_with_mean_std(
    merged=merged,
    all_series=all_series,
    x=x,
    hist_steps=hist_steps,
    gen_block=gen_block,
    num_insertions=num_insertions,
    num_coolings=num_coolings,
    n_gen_msgs=n_gen_msgs,
    midprice_step_size=midprice_step_size,
)
fig.show()

insertion positions: [551, 602, 653, 704, 755, 806, 857, 908, 959, 1010, 1061, 1112, 1163, 1214, 1265, 1316, 1367, 1418, 1469, 1520]
cooling positions:   [1570, 1620, 1670, 1720, 1770, 1820, 1870, 1920, 1970, 2020, 2070, 2120, 2170, 2220, 2270, 2320, 2370, 2420, 2470, 2520, 2570, 2620, 2670, 2720, 2770, 2820, 2870, 2920, 2970, 3020]


In [9]:
# np.save("/app/data_saved/exp_96_20250703_212149/exp_96_20250703_212149_mean.npy", mean_series)
# np.save("/app/data_saved/exp_96_20250703_212149/exp_96_20250703_212149_std.npy", std_series)

# ----------------------------------------------------

# OREDER-PLAYER

In [10]:
import os, glob, re
import numpy as np
import pandas as pd

def build_and_merge(folder, batch_prefix, inp_prefix):
    # STEP 1: load every .npy (shape (batch_size, time, feat)) into a DataFrame
    files   = glob.glob(os.path.join(folder, "*.npy"))
    rx_iter = re.compile(rf"{re.escape(batch_prefix)}_\[(.+)\]_iter_(\d+)\.npy$")
    rx_inp  = re.compile(rf"{re.escape(inp_prefix)}_\[(.+)\]\.npy$")
    rec = []
    for f in files:
        nm = os.path.basename(f)
        m  = rx_iter.match(nm)
        if m:
            rng, itr = m.group(1).replace(" ", ""), int(m.group(2))
        else:
            m2 = rx_inp.match(nm)
            if not m2:
                continue
            rng, itr = m2.group(1).replace(" ", ""), 0

        batch = np.load(f)  # shape (batch_size, time, features)
        print(f"Loaded {nm} with shape {batch.shape}")

        rec.append({"range": rng, "iteration": itr, "batch": batch})
    df = pd.DataFrame(rec).sort_values(["range","iteration"]).reset_index(drop=True)

    # STEP 2: parse the comma‐separated list of IDs into Python lists
    df["ids"] = df["range"].str.split(",").apply(lambda L: [int(x) for x in L])

    # explode each batch into one row per sample, с учётом slicing
    rows = []
    for _, r in df.iterrows():
        for idx, sample_id in enumerate(r["ids"]):
            single = r["batch"][idx]   # shape (time, features)

            # ====== здесь происходит нужный slice ======
            if r["iteration"] > 0:
                n_keep = 51 if r["iteration"] <= num_insertions else 50
                single = single[-n_keep:, :]
            # ============================================

            rows.append({
                "id":        sample_id,
                "iteration": r["iteration"],
                "data":      single
            })

    df_sorted = pd.DataFrame(rows).sort_values(["id","iteration"]).reset_index(drop=True)

    merged = []
    for id_val, grp in df_sorted.groupby("id", sort=True):
        arrs = [row.data for _, row in grp.iterrows()]
        big  = np.concatenate(arrs, axis=0)
        merged.append({"id": id_val, "merged_data": big})
    merged_df = pd.DataFrame(merged).sort_values("id").reset_index(drop=True)

    return df, df_sorted, merged_df

b_folder      = f"/app/data_saved/{experiment_name}/b_seq_gen_doubled"
b_batch_pref  = "b_seq_gen_doubled_batch"
b_inp_pref    = "b_seq_inp"

m_folder      = f"/app/data_saved/{experiment_name}/msgs_decoded_doubled"
m_batch_pref  = "msgs_decoded_doubled_batch"
m_inp_pref    = "m_seq_raw_inp"

_, b_sorted, b_merged = build_and_merge(b_folder, b_batch_pref, b_inp_pref)
_, m_sorted, m_merged = build_and_merge(m_folder, m_batch_pref, m_inp_pref)

b_dict = { int(r.id): np.array(r.merged_data) for _, r in b_merged.iterrows() }
m_dict = { int(r.id): np.array(r.merged_data) for _, r in m_merged.iterrows() }

for d in (b_dict, m_dict):
    for key, arr in d.items():
        zero = np.zeros((1, arr.shape[1]), dtype=arr.dtype)
        d[key] = np.vstack([zero, arr])

Loaded b_seq_gen_doubled_batch_[22181, 1895, 36336, 11828]_iter_3.npy with shape (4, 501, 501)
Loaded b_seq_gen_doubled_batch_[22181, 1895, 36336, 11828]_iter_23.npy with shape (4, 500, 501)
Loaded b_seq_gen_doubled_batch_[32404, 19669, 16941, 21490]_iter_30.npy with shape (4, 500, 501)
Loaded b_seq_gen_doubled_batch_[14606, 16120, 2787, 4668]_iter_22.npy with shape (4, 500, 501)
Loaded b_seq_gen_doubled_batch_[35217, 6779, 27531, 21001]_iter_18.npy with shape (4, 501, 501)
Loaded b_seq_gen_doubled_batch_[23105, 8306, 3489, 26578]_iter_39.npy with shape (4, 500, 501)
Loaded b_seq_gen_doubled_batch_[20394, 10954, 4447, 32904]_iter_3.npy with shape (4, 501, 501)
Loaded b_seq_gen_doubled_batch_[20394, 10954, 4447, 32904]_iter_14.npy with shape (4, 501, 501)
Loaded b_seq_gen_doubled_batch_[23718, 20774, 33058, 37341]_iter_43.npy with shape (4, 500, 501)
Loaded b_seq_gen_doubled_batch_[16538, 8604, 6556, 38336]_iter_4.npy with shape (4, 501, 501)
Loaded b_seq_gen_doubled_batch_[35217, 6779,

In [11]:
# # Save m_dict to a pickle file for later use
# import pickle

# with open('m_dict.pkl', 'wb') as f:
#     pickle.dump(m_dict, f)

# print(f"m_dict saved with {len(m_dict)} samples")



In [12]:
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

def interactive_lob_plot(b_seq_inp, msg_seq_raw):
    # allow DataFrame or dict
    if isinstance(b_seq_inp, pd.DataFrame):
        b_seq_inp = {int(r.id): np.array(r.merged_data) for _,r in b_seq_inp.iterrows()}
    if isinstance(msg_seq_raw, pd.DataFrame):
        msg_seq_raw = {int(r.id): np.array(r.merged_data) for _,r in msg_seq_raw.iterrows()}

    # controls
    id_dd       = widgets.Dropdown(options=sorted(b_seq_inp.keys()), description="Sample ID:")
    time_slider = widgets.IntSlider(min=1, max=1, step=1, description="t:")
    btn_prev    = widgets.Button(description="←")
    btn_next    = widgets.Button(description="→")
    msg_box     = widgets.HTML()

    # figure with two subplots
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Book state t–1","Book state t"])
    fig.add_trace(go.Bar(x=[],y=[]), row=1,col=1)
    fig.add_trace(go.Bar(x=[],y=[]), row=1,col=2)
    fig.update_layout(width=800, height=400, showlegend=False, template='plotly_white')
    fig_widget = go.FigureWidget(fig)

    def update_slider_range(*_):
        arr = b_seq_inp[id_dd.value]
        
        time_slider.min = 1
        time_slider.max = arr.shape[0] - 1
        time_slider.value = 1

    def update_plot(*_):
        sid = id_dd.value
        t   = time_slider.value
        arr = b_seq_inp[sid]
        msgs= msg_seq_raw[sid]

        
        s0 = arr[t-1, 240:263]
        s1 = arr[t,   240:263]

        diff = abs(s1) - abs(s0)
        x = np.arange(len(s0)) - len(s0)//2

        with fig_widget.batch_update():
            fig_widget.data = []
            fig_widget.add_bar(x=x, y=s0, row=1, col=1, marker_color='orange')
            
            colors = ['orange' if abs(d)<1e-8 else ('red' if d>0 else 'blue') for d in diff]
                        
            fig_widget.add_bar(x=x, y=s1, row=1, col=2, marker_color=colors)
            fig_widget.layout.annotations[0].text = f"Book state {t-1}"
            fig_widget.layout.annotations[1].text = f"Book state {t}"

        # show message at index t
        m = msgs[t].astype(int)
        # fields: [0]=timestamp, [1]=etype, [2]=dir, [3]=abspr, [4]=relpr, [5]=size, …
        et, dr, abspr, relpr, sz = m[1], m[2], m[3], m[4], m[5]
        et_map = {1:"Limit",2:"PartialCancel",3:"Delete",4:"Execution"}
        dr_map = {1:"Buy",0:"Sell"}
        info = (
            f"{et_map.get(et,'?')} • "
            f"{dr_map.get(dr,'?')} • "
            f"abs={abspr} • rel={relpr} • size={sz}"
        )
        msg_box.value = f"<b>{info}<br></b>raw:{m.tolist()}"

    def on_prev(b):
        if time_slider.value>time_slider.min:
            time_slider.value -= 1
    def on_next(b):
        if time_slider.value<time_slider.max:
            time_slider.value += 1

    # wire up events
    id_dd.observe(lambda c: update_slider_range(), names='value')
    time_slider.observe(lambda c: update_plot(), names='value')
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)

    # initial draw
    update_slider_range()
    update_plot()

    display(widgets.HBox([id_dd, btn_prev, btn_next, time_slider]))
    display(fig_widget, msg_box)


interactive_lob_plot(b_dict, m_dict)

FigureWidget({
    'data': [{'marker': {'color': 'orange'},
              'type': 'bar',
              'uid': 'd9a2b285-5de3-4d60-95d5-03bf70fde332',
              'x': {'bdata': '9fb3+Pn6+/z9/v8AAQIDBAUGBwgJCgs=', 'dtype': 'i1'},
              'xaxis': 'x',
              'y': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAA='),
                    'dtype': 'f4'},
              'yaxis': 'y'},
             {'marker': {'color': [red, red, red, red, red, red, red, red, red,
                                   red, orange, orange, orange, red, red, red, red,
                                   red, red, red, red, red, red]},
              'type': 'bar',
              'uid': '502d9832-7613-404c-a32d-b3217889e6e1',
              'x': {'bdata': '9fb3+Pn6+/z9/v8AAQIDBAUGBwgJCgs=', 'dtype': 'i1'},
              'xaxis': 'x2',
              'y': {'bdata': ('XY+CPpqZGT5XDk0/AAAAP83MTD4w3a' ... 'O/bxKDvcl2vr5KDLK/C9ejvpDCdb4='),
                    'dtype': 'f4'},
     

HTML(value='<b>Limit • Buy • abs=903800 • rel=-3 • size=200<br></b>raw:[39480138, 1, 1, 903800, -3, 200, 0, 34…

In [13]:
def show_execution_indices(m_dict):
    """
    For each sample_id in m_dict, prints indices t where etype == 4 (Execution).
    """
    for sid, msgs in m_dict.items():
        etypes = msgs[:, 1]
        exec_indices = np.where(etypes == 4)[0]
        if len(exec_indices) > 0:
            print(f"Sample ID {sid}: Execution at indices {exec_indices.tolist()}")
        else:
            print(f"Sample ID {sid}: No Executions found")

show_execution_indices(m_dict)

Sample ID 114: Execution at indices [130, 148, 149, 551, 602, 639, 646, 653, 690, 697, 704, 741, 748, 755, 806, 857, 908, 945, 952, 953, 959, 1010]
Sample ID 648: Execution at indices [3, 37, 156, 157, 162, 181, 346, 347, 352, 353, 487, 551, 602, 653, 703, 704, 755, 806, 857, 907, 908, 959, 1009, 1010]
Sample ID 1895: Execution at indices [164, 165, 225, 232, 353, 480, 484, 485, 551, 602, 653, 704, 726, 727, 728, 729, 755, 777, 778, 779, 780, 806, 828, 829, 830, 831, 857, 879, 880, 881, 882, 908, 959, 981, 982, 983, 984, 1010, 1032, 1033, 1034, 1035, 1061, 1083, 1084, 1085, 1086, 1112, 1134, 1135, 1136, 1137, 1163, 1185, 1186, 1187, 1188, 1214, 1236, 1237, 1238, 1239, 1265, 1287, 1288, 1289, 1290, 1316, 1338, 1339, 1340, 1341, 1367, 1389, 1390, 1391, 1392, 1418, 1440, 1441, 1442, 1443, 1469, 1491, 1492, 1493, 1494, 1520, 1542, 1543, 1544, 1545, 1592, 1593, 1594, 1595, 1642, 1643, 1644, 1645, 1692, 1693, 1694, 1695, 1742, 1743, 1744, 1745, 1792, 1793, 1794, 1795, 1842, 1843, 1844, 1845,

# Market Impact graph

In [14]:
from IPython.display import display
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import ipywidgets as widgets


def interactive_market_impact_plot(
    b_seq_inp,            # dict[id] -> np.array(T, ...), где столбец 0 = Δmid (в тиках/шаги)
    msg_seq_raw,          # dict[id] -> np.array(T, num_fields)
    all_series,           # не обязателен для mid теперь; можно передать список-заглушку
    x,                    # ось времени для средней цены (любой той же длины, что T)
    hist_steps=550,
    gen_block=50,
    num_insertions=20,
    tick_size=100,        # 1 тик = 0.01$ -> 100 если цены в центах
):
    """
    • Q накапливается ТОЛЬКО по нашим вставкам (с нуля).
    • V_exp: кумулятивный объём рыночных исполнений (event_type==4) от индекса 0 до (idx-1).
    • Impact = |VWAP_inserted - reference_price_ticks| (в тиках).
    • Абсолютный mid восстанавливаем из кумулятивной суммы Δmid (столбец 0 книги),
      привязав его к цене ссылки на первой вставке.
    """

    # Приведение входов из DataFrame -> dict[int] -> np.array
    if isinstance(b_seq_inp, pd.DataFrame):
        b_seq_inp = {int(r.id): np.array(r.merged_data) for _, r in b_seq_inp.iterrows()}
    if isinstance(msg_seq_raw, pd.DataFrame):
        msg_seq_raw = {int(r.id): np.array(r.merged_data) for _, r in msg_seq_raw.iterrows()}

    # --- UI ---
    id_dd       = widgets.Dropdown(options=sorted(b_seq_inp.keys()), description="Sample ID:")
    time_slider = widgets.IntSlider(min=1, max=1, step=1, description="t:")
    btn_prev    = widgets.Button(description="←")
    btn_next    = widgets.Button(description="→")
    msg_box     = widgets.HTML()
    coeff_box   = widgets.HTML()

    if 79 in b_seq_inp:
        id_dd.value = 79

    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=["Book state (ΔL2)", "Mid (absolute ticks)", "Market Impact (log-log)"])
    fig.update_layout(width=1400, height=520, showlegend=True, template='plotly_white')
    fig_widget = go.FigureWidget(fig)

    # индексы колонок в raw-сообщении (подстрои под свои)
    EVENT_TYPE_COL = 1
    DIRECTION_COL  = 2
    PRICE_COL      = 3   # абсолютная цена (в тиках/центах) для вставок
    REL_COL        = 4
    SIZE_COL       = 5

    def update_slider_range(*_):
        arr = b_seq_inp[id_dd.value]
        time_slider.min = 1
        time_slider.max = arr.shape[0] - 1
        time_slider.value = min(551, time_slider.max)

    def update_plot(*_):
        sample_id = id_dd.value
        t = time_slider.value

        book_array = b_seq_inp[sample_id]      # [T, ...]
        messages   = msg_seq_raw[sample_id]    # [T, num_fields]
        T = len(messages)

        # ----- (1) Book viz: сравним L2 на t-1 и t (тут сохраняю твою логику) -----
        # В примерах L2 был в диапазоне [240:263], здесь оставлю по умолчанию.
        # При необходимости поправь слайс под свою схему.
        l2_slice = slice(240, 263)
        book_prev = book_array[t-1, l2_slice]
        book_now  = book_array[t,   l2_slice]
        book_diff = np.abs(book_now) - np.abs(book_prev)
        x_levels  = np.arange(len(book_prev)) - len(book_prev)//2
        book_colors = ['orange' if abs(d) < 1e-8 else ('red' if d > 0 else 'blue') for d in book_diff]

        # ----- (2) Позиции вставок -----
        insertion_positions = hist_steps + np.arange(1, num_insertions + 1) * gen_block
        valid_insertions = [pos for pos in insertion_positions if pos < T]
        if not valid_insertions:
            coeff_box.value = "<b>No valid insertions.</b>"
            with fig_widget.batch_update():
                fig_widget.data = []
                fig_widget.layout.shapes = []
            return

        # ----- (3) Базовая цена ссылки (абсолютные тики) на первой вставке -----
        ref_idx = valid_insertions[0]
        reference_price_ticks = float(messages[ref_idx, PRICE_COL])

        # ----- (4) Q_i, VWAP_i по вставкам (в абсолютных тиках) -----
        insert_sizes  = messages[valid_insertions, SIZE_COL].astype(float)     # ΔQ_i
        insert_prices = messages[valid_insertions, PRICE_COL].astype(float)    # P_i (абс. тики)
        Q_deltas = insert_sizes
        Q_cum    = np.cumsum(Q_deltas)
        notional_ticks_cum = np.cumsum(Q_deltas * insert_prices)
        vwap_ticks_series  = notional_ticks_cum / np.maximum(Q_cum, 1e-12)
        impact_ticks       = np.abs(vwap_ticks_series - reference_price_ticks)

        # ----- (5) V_exp до (idx-1) от старта -----
        evt_types   = messages[:, EVENT_TYPE_COL].astype(int)
        exec_sizes  = np.where(evt_types == 4, messages[:, SIZE_COL].astype(float), 0.0)
        cum_exec_vol = np.cumsum(exec_sizes)
        V_exp = np.array([float(cum_exec_vol[idx-1] if (idx-1) >= 0 else 0.0)
                          for idx in valid_insertions])

        # ----- (6) Лог-преобразования -----
        eps = 1e-12
        rel_size    = Q_cum / np.maximum(V_exp, eps)
        log_qv      = np.log(np.maximum(rel_size, eps))
        log_impact  = np.log(np.maximum(impact_ticks, eps))

        # ----- (7) ВОССТАНОВЛЕНИЕ АБСОЛЮТНОГО MID -----
        # Предполагаем, что столбец 0 книги = Δmid_t (изменение mid в тиках между t-1 и t).
        # Тогда cum_Δmid = cumsum(Δmid), и привязываем абсолют к reference_price_ticks на ref_idx.
        delta_mid_ticks = book_array[:, 0].astype(float)       # Δmid_t в тиках
        cum_delta_mid   = np.cumsum(delta_mid_ticks)           # накопленная Δmid от старта окна
        # Абсолютный mid в тиках, привязанный к ref:
        mid_abs_ticks   = reference_price_ticks + (cum_delta_mid - cum_delta_mid[ref_idx]) * tick_size

        # Для отрисовки временного ряда mid используем mid_abs_ticks:
        mid_series_abs = mid_abs_ticks

        # ----- (8) Подробный лог (теперь всё в абсолютных тиках) -----
        print("\n=== Step-by-step X (log(Q/V_exp)) and Y (price/impact path) ===")
        for i, idx in enumerate(valid_insertions):
            Q_i, V_i = Q_cum[i], V_exp[i]
            x_raw = Q_i / max(V_i, eps)
            x_log = np.log(max(x_raw, eps))

            mid_ref_abs = float(mid_series_abs[ref_idx])
            mid_i_abs   = float(mid_series_abs[idx]) if idx < len(mid_series_abs) else np.nan
            dmid_ticks  = (mid_i_abs - mid_ref_abs) / tick_size

            vwap_i      = float(vwap_ticks_series[i])
            impact_i    = float(impact_ticks[i])
            y_log       = float(log_impact[i])

            # X-строка
            print(f"[{i+1}] X = log(Q/V_exp) = log({Q_i:.4f} / {V_i:.4f}) = {x_log:.8f}")
            # Y-строка: все mid в абсолютных тиках; Δmid в тиках
            print(
                f"    Y: mid_ref={mid_ref_abs:.2f}, mid_i={mid_i_abs:.2f}, Δmid={dmid_ticks:.2f} ticks; "
                f"Impact = |VWAP - ref_price| = |{vwap_i:.2f} - {reference_price_ticks:.2f}| = {impact_i:.6f}; "
                f"log(Impact)={y_log:.8f}"
            )
            print("")

        # ----- (9) Регрессия -----
        X = log_qv.reshape(-1, 1)
        y = log_impact
        reg = LinearRegression().fit(X, y)
        delta_hat = float(reg.coef_[0])
        log_lambda_hat = float(reg.intercept_)
        beta_theory = 0.5

        coeff_box.value = f"""
        <div style="padding-left:20px; font-family:monospace">
            <h4>Market Impact (Absolute ticks; V_exp from 0)</h4>
            <p><b>Estimated:</b> log(Impact) = <b>{log_lambda_hat:.4f}</b> + <b>{delta_hat:.4f}</b> · log(Q / V_exp)</p>
            <p><b>Theoretical:</b> β = {beta_theory:.2f}</p>
        </div>
        """

        # Линии регрессии
        ref_line_x = np.linspace(float(log_qv.min()), float(log_qv.max()), 100)
        ref_line_y = log_lambda_hat + delta_hat * ref_line_x
        theory_line_y = log_lambda_hat + beta_theory * ref_line_x

        # Метки для вставок (1..N)
        insert_labels = [str(i) for i, _ in enumerate(valid_insertions, start=1)]

        # ----- (10) Рисуем -----
        with fig_widget.batch_update():
            fig_widget.data = []
            fig_widget.layout.shapes = []

            # (1) L2 book bars
            fig_widget.add_bar(x=x_levels, y=book_prev, row=1, col=1, marker_color='orange', name="Prev L2")
            fig_widget.add_bar(x=x_levels, y=book_now,  row=1, col=1, marker_color=book_colors, name="Now L2")

            # (2) mid (absolute ticks)
            fig_widget.add_scatter(x=np.arange(T), y=mid_series_abs, mode='lines',
                                   row=1, col=2, line=dict(width=1), name="Mid (abs ticks)")
            fig_widget.add_shape(type="line", x0=t, x1=t,
                                 y0=float(np.nanmin(mid_series_abs)),
                                 y1=float(np.nanmax(mid_series_abs)),
                                 line=dict(color="green", width=2), xref="x2", yref="y2")

            # insertion markers на mid (hover: 1..N)
            insert_dots_x = [p for p in valid_insertions if p < len(mid_series_abs)]
            insert_dots_y = [mid_series_abs[p] for p in insert_dots_x]
            fig_widget.add_scatter(
                x=insert_dots_x,
                y=insert_dots_y,
                mode='markers',
                marker=dict(size=7, symbol='circle'),
                text=insert_labels[:len(insert_dots_x)],
                hovertemplate="Insertion %{text}<extra></extra>",
                row=1, col=2,
                name="Insert marks"
            )

            # (3) log-log точки
            fig_widget.add_scatter(
                x=log_qv, y=log_impact, mode='markers',
                marker=dict(size=8),
                text=insert_labels,
                hovertemplate="Insertion %{text}<extra></extra>",
                row=1, col=3, name="Points"
            )
            fig_widget.add_scatter(x=ref_line_x, y=ref_line_y, mode='lines',
                                   line=dict(dash='dot', width=2), row=1, col=3, name="Estimated")
            fig_widget.add_scatter(x=ref_line_x, y=theory_line_y, mode='lines',
                                   line=dict(dash='dash', width=2), row=1, col=3, name="Theoretical β=0.5")

            # Подсветка текущей вставки
            if t in valid_insertions:
                i_sel = valid_insertions.index(t)
                fig_widget.add_scatter(
                    x=[log_qv[i_sel]], y=[log_impact[i_sel]], mode='markers',
                    marker=dict(color='red', size=12),
                    text=[insert_labels[i_sel]],
                    hovertemplate="Insertion %{text}<extra></extra>",
                    row=1, col=3, name="Current"
                )

            fig_widget.update_xaxes(title="log(Q / V_exp)", row=1, col=3)
            fig_widget.update_yaxes(title="log(Impact)",   row=1, col=3)

        # raw message info на t
        m = messages[t].astype(int)
        event_map = {1: "Limit", 2: "PartialCancel", 3: "Delete", 4: "Execution"}
        direction_map = {1: "Buy", 0: "Sell"}
        msg_box.value = (
            f"<b>{event_map.get(m[EVENT_TYPE_COL], '?')} • {direction_map.get(m[DIRECTION_COL], '?')} "
            f"• abs={m[PRICE_COL]} • rel={m[REL_COL]} • size={m[SIZE_COL]}</b><br>raw: {m.tolist()}"
        )

    def on_prev(_):
        if time_slider.value > time_slider.min:
            time_slider.value -= 1

    def on_next(_):
        if time_slider.value < time_slider.max:
            time_slider.value += 1

    id_dd.observe(lambda _: update_slider_range(), names='value')
    time_slider.observe(lambda _: update_plot(), names='value')
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)

    update_slider_range()
    update_plot()

    display(widgets.HBox([id_dd, btn_prev, btn_next, time_slider]))
    display(widgets.HBox([fig_widget, coeff_box]))
    display(msg_box)


In [15]:
# interactive_market_impact_plot(b_dict, m_dict, all_series, x)

all_betas = interactive_market_impact_plot(b_dict, m_dict, all_series, x, hist_steps, gen_block, num_insertions)


=== Step-by-step X (log(Q/V_exp)) and Y (price/impact path) ===
[1] X = log(Q/V_exp) = log(110.0000 / 508.0000) = -1.53000108
    Y: mid_ref=904500.00, mid_i=904500.00, Δmid=0.00 ticks; Impact = |VWAP - ref_price| = |904500.00 - 904500.00| = 0.000000; log(Impact)=-27.63102112

[2] X = log(Q/V_exp) = log(115.0000 / 618.0000) = -1.68155633
    Y: mid_ref=904500.00, mid_i=904600.00, Δmid=1.00 ticks; Impact = |VWAP - ref_price| = |904504.35 - 904500.00| = 4.347826; log(Impact)=1.46967597

[3] X = log(Q/V_exp) = log(728.0000 / 663.0000) = 0.09352606
    Y: mid_ref=904500.00, mid_i=904700.00, Δmid=2.00 ticks; Impact = |VWAP - ref_price| = |904669.09 - 904500.00| = 169.093407; log(Impact)=5.13045126

[4] X = log(Q/V_exp) = log(963.0000 / 1319.0000) = -0.31457574
    Y: mid_ref=904500.00, mid_i=905000.00, Δmid=5.00 ticks; Impact = |VWAP - ref_price| = |904725.44 - 904500.00| = 225.441329; log(Impact)=5.41805994

[5] X = log(Q/V_exp) = log(2012.0000 / 1604.0000) = 0.22662874
    Y: mid_ref=904

    'data': [{'marker': {'color': 'orange'},
              'name': 'Prev L2',
  …

HTML(value='<b>Execution • Sell • abs=904500 • rel=1 • size=110</b><br>raw: [77777777, 4, 0, 904500, 1, 110, 0…

# FILTRADE IF IMPACT != 0

In [16]:
sample_day_map.shape

(1024, 5)

In [17]:
from IPython.display import display
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import ipywidgets as widgets


def calculate_impact(messages, valid_insertions, reference_price):
    """
    Calculate market impact for each insertion.
    
    Parameters
    ----------
    messages : np.array
        Message array with columns [EVENT_TYPE, DIRECTION, PRICE, REL, SIZE, ...]
    valid_insertions : list
        List of insertion indices
    reference_price : float
        Reference price at first insertion
        
    Returns
    -------
    impact : np.array
        Absolute impact for each insertion
    vwap_series : np.array
        VWAP series for each insertion
    Q_cum : np.array
        Cumulative quantity for each insertion
    log_imp : np.array
        Log of impact values for plotting
    """
    PRICE_COL = 3
    SIZE_COL = 5
    insert_sizes = messages[valid_insertions, SIZE_COL].astype(float)
    insert_prices = messages[valid_insertions, PRICE_COL].astype(float)
    Q_cum = np.cumsum(insert_sizes)
    notional = np.cumsum(insert_sizes * insert_prices)

    vwap_series = notional / np.maximum(Q_cum, 1e-12)
    # vwap_series = insert_prices
    
    impact = np.abs(vwap_series - reference_price) / reference_price
    
    # Calculate log impact for plotting (y-axis)
    eps = 1e-12
    log_imp = np.log(np.maximum(impact, eps))
    
    return impact, vwap_series, Q_cum, log_imp


def calculate_market_volume(messages, hist_steps, valid_insertions, execution_sum):
    """
    Calculate market execution volume V_exp for each insertion and return x-axis values for plotting.
    
    Parameters
    ----------
    messages : np.array
        Message array with columns [EVENT_TYPE, DIRECTION, PRICE, REL, SIZE, ...]
    hist_steps : int
        Starting index for market volume calculation
    valid_insertions : list
        List of insertion indices
    execution_sum : float
        Total execution sum for the day
        
    Returns
    -------
    V_exp : np.array
        Market volume from hist_steps to (idx-1) for each insertion
    log_qv : np.array
        Log of Q/V_exp ratio for plotting (x-axis)
    """
    EVENT_TYPE_COL = 1
    SIZE_COL = 5
    
    evt_types = messages[:, EVENT_TYPE_COL].astype(int)
    exec_sizes = np.where(evt_types == 4, messages[:, SIZE_COL].astype(float), 0.0)
    cum_exec_vol = np.cumsum(exec_sizes)
    
    V_exp = np.array([float(cum_exec_vol[idx-1] - cum_exec_vol[hist_steps-1] if (idx-1) >= hist_steps else 0.0)
                      for idx in valid_insertions])

    # Use execution_sum instead of fixed value
    V_exp = np.full_like(V_exp, execution_sum)
    
    # Calculate cumulative quantity for Q/V_exp ratio
    insert_sizes = messages[valid_insertions, SIZE_COL].astype(float)
    Q_cum = np.cumsum(insert_sizes)
    
    # Calculate log(Q/V_exp) for plotting (x-axis)
    eps = 1e-12
    rel_size = Q_cum / np.maximum(V_exp, eps)
    log_qv = np.log(np.maximum(rel_size, eps))

    return V_exp, log_qv


def interactive_market_impact_plot(
    b_seq_inp,            # dict[id] -> np.array(T, ...), where col 0 = Δmid per step
    msg_seq_raw,          # dict[id] -> np.array(T, num_fields)
    all_series,           # unused for mid now
    x,                    # time axis (len T)
    hist_steps=550,
    gen_block=50,
    num_insertions=20,
    beta_theory=0.5,      # theoretical slope
    tick_size=100,        # tick size for price conversion
):
    """
    • Q accumulates ONLY our insertions (from zero).
    • V_exp: cumulative market executions (event_type==4) from index hist_steps to (idx-1).
    • Impact = |VWAP_inserted - reference_price|.
    • Absolute mid reconstructed from cumulative Δmid (book[:,0]), anchored to ref price at first insertion.
    • In log–log panel: raw log values without normalization.
      - Allowed points (used for fit): colored
      - Zero-impact points: grey, excluded from fit
    • Theoretical line: passes through fixed intercept with slope beta_theory.
    • Prices converted from ticks to dollars using tick_size.
    """

    # Accept DataFrame inputs -> dict[int] -> np.array
    if isinstance(b_seq_inp, pd.DataFrame):
        b_seq_inp = {int(r.id): np.array(r.merged_data) for _, r in b_seq_inp.iterrows()}
    if isinstance(msg_seq_raw, pd.DataFrame):
        msg_seq_raw = {int(r.id): np.array(r.merged_data) for _, r in msg_seq_raw.iterrows()}

    # --- UI ---
    id_dd       = widgets.Dropdown(options=sorted(b_seq_inp.keys()), description="Sample ID:")
    time_slider = widgets.IntSlider(min=1, max=1, step=1, description="t:")
    btn_prev    = widgets.Button(description="←")
    btn_next    = widgets.Button(description="→")
    msg_box     = widgets.HTML()
    coeff_box   = widgets.HTML()

    if 79 in b_seq_inp:
        id_dd.value = 79

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=["Book state (ΔL2)", "Mid (absolute)", "Market Impact (log–log)"]
    )
    fig.update_layout(width=1400, height=520, showlegend=True, template='plotly_white')
    fig_widget = go.FigureWidget(fig)

    # raw message column indices (adjust if needed)
    EVENT_TYPE_COL = 1
    DIRECTION_COL  = 2
    PRICE_COL      = 3   # absolute price in ticks
    REL_COL        = 4
    SIZE_COL       = 5

    def update_slider_range(*_):
        arr = b_seq_inp[id_dd.value]
        time_slider.min = 1
        time_slider.max = arr.shape[0] - 1
        time_slider.value = min(551, time_slider.max)

    def update_plot(*_):
        sample_id = id_dd.value
        t = time_slider.value

        book_array = b_seq_inp[sample_id]      # [T, ...]
        messages   = msg_seq_raw[sample_id]    # [T, num_fields]
        T = len(messages)

        # Get day data for this sample from sample_day_map
        sample_row = sample_day_map[sample_day_map['sample_id'] == sample_id]
        if sample_row.empty:
            coeff_box.value = f"<b>Sample {sample_id} not found in sample_day_map.</b>"
            with fig_widget.batch_update():
                fig_widget.data = []
                fig_widget.layout.shapes = []
            return
        
        highest_price = sample_row.iloc[0]['highest_price']
        lowest_price = sample_row.iloc[0]['lowest_price']
        execution_sum = sample_row.iloc[0]['execution_sum']
        
        # (1) Book viz (ΔL2 slices)
        # l2_slice = slice(240, 263)  # adjust to your layout
        l2_slice = slice(20, 800)  # adjust to your layout
        book_prev = book_array[t-1, l2_slice]
        book_now  = book_array[t,   l2_slice]
        book_diff = np.abs(book_now) - np.abs(book_prev)
        x_lvls    = np.arange(len(book_prev)) - len(book_prev)//2
        book_cols = ['orange' if abs(d) < 1e-8 else ('red' if d > 0 else 'blue') for d in book_diff]

        # (2) Insertion positions
        insertion_positions = hist_steps + np.arange(1, num_insertions + 1) * gen_block
        valid_insertions = [pos for pos in insertion_positions if pos < T]
        if not valid_insertions:
            coeff_box.value = "<b>No valid insertions.</b>"
            with fig_widget.batch_update():
                fig_widget.data = []
                fig_widget.layout.shapes = []
            return

        # (3) Reference absolute price at first insertion (convert from ticks to dollars)
        ref_idx = valid_insertions[0]
        reference_price = float(messages[ref_idx, PRICE_COL]) / tick_size

        # (4) Absolute mid reconstruction (for middle chart and eta calculation)
        delta_mid = book_array[:, 0].astype(float) / tick_size  # Δmid_t in dollars
        cum_delta_mid   = np.cumsum(delta_mid)
        mid_abs   = reference_price + (cum_delta_mid - cum_delta_mid[ref_idx])
        mid_series_abs  = mid_abs

        # (5) Calculate impact and market volume using helper functions - get x and y for plotting
        # Convert prices to dollars in the helper functions
        messages_dollars = messages.copy().astype(float)
        messages_dollars[:, PRICE_COL] /= tick_size
        
        impact, vwap_series, Q_cum, log_imp = calculate_impact(messages_dollars, valid_insertions, reference_price)
        V_exp, log_qv = calculate_market_volume(messages_dollars, hist_steps, valid_insertions, execution_sum)

        # (6) Calculate eta for display using highest_price and lowest_price from day data
        H = highest_price / tick_size  # Convert to dollars
        L = lowest_price / tick_size   # Convert to dollars
        print(f"DEBUG eta calculation: H={H:.10f}, L={L:.10f}")
        
        if H > L:
            ln_hl = np.log(H / L)
            print(f"DEBUG eta calculation: ln(H/L)={ln_hl:.10f}")
            eta_day = ln_hl / 0.8325546
            print(f"DEBUG eta calculation: eta_day = {ln_hl:.10f} / 0.8325546 = {eta_day:.10f}")
            print(f"DEBUG ln(eta) calculation: ln(eta_day) = {np.log(eta_day):.10f}")
        else:
            eta_day = 1e-12
            print(f"DEBUG eta calculation: H <= L, using eta_day = {eta_day:.10f}")

        # (7) Determine which points to use for fitting
        tol = 1e-12
        mask_zero = impact <= tol
        mask_pos  = ~mask_zero

        # (8) Fit on allowed points
        used_x_raw = log_qv[mask_pos]
        used_y_raw = log_imp[mask_pos]

        if used_x_raw.size >= 2:
            # Fit in raw log space
            # Calculate fixed intercept based on high/low of midprices
            fixed_intercept = np.log(eta_day)
            
            # Fit regression with fixed intercept
            # y = fixed_intercept + beta * x, so we solve for beta using: beta = mean((y - fixed_intercept) / x)
            adjusted_y = used_y_raw - fixed_intercept
            beta_hat = float(np.mean(adjusted_y / used_x_raw))
            alpha_hat = fixed_intercept

            # Calculate min/max and deltas for allowed points only
            x_min, x_max = float(used_x_raw.min()), float(used_x_raw.max())
            y_min, y_max = float(used_y_raw.min()), float(used_y_raw.max())
            x_delta = x_max - x_min
            y_delta = y_max - y_min

            # Theoretical line in raw log space: y = fixed_intercept + beta_theory * x
            n_used = used_x_raw.shape[0]; n_total = len(log_qv)
            coeff_box.value = f"""
            <div style="padding-left:20px; font-family:monospace">
                <h4>Market Impact (log–log; allowed points only) - Sample {sample_id}</h4>
                <p><b>Fit:</b> log(Impact) = <b>{alpha_hat:.10f}</b> + <b>{beta_hat:.10f}</b> · log(Q/V_exp)</p>
                <p><b>Theory:</b> log(Impact) = {fixed_intercept:.10f} + {beta_theory:.2f} · log(Q/V_exp)</p>
                <p><b>Eta:</b> {eta_day:.10f}</p>
                <p><b>H:</b> ${H:.2f}, <b>L:</b> ${L:.2f}, <b>Execution Sum:</b> {execution_sum:.0f}</p>
                <p>Used points: {n_used} / {n_total}</p>
                <p><b>X range:</b> [{x_min:.10f}, {x_max:.10f}] (Δ={x_delta:.10f})</p>
                <p><b>Y range:</b> [{y_min:.10f}, {y_max:.10f}] (Δ={y_delta:.10f})</p>
            </div>
            """

            # Line ranges for plotting
            xspan = np.linspace(float(used_x_raw.min()), float(used_x_raw.max()), 100)
            fit_y = alpha_hat + beta_hat * xspan
            th_y  = fixed_intercept + beta_theory * xspan

        else:
            xspan = np.array([]); fit_y = np.array([]); th_y = np.array([])
            coeff_box.value = f"""
            <div style="padding-left:20px; font-family:monospace">
                <h4>Market Impact (log–log) - Sample {sample_id}</h4>
                <p><b>Fit:</b> not enough non-zero impact points.</p>
                <p><b>Eta:</b> {eta_day:.10f}</p>
                <p><b>H:</b> ${H:.2f}, <b>L:</b> ${L:.2f}, <b>Execution Sum:</b> {execution_sum:.0f}</p>
            </div>
            """

        # Labels 1..N
        insert_labels = [str(i) for i, _ in enumerate(valid_insertions, start=1)]

        # (9) Draw
        with fig_widget.batch_update():
            fig_widget.data = []
            fig_widget.layout.shapes = []

            # (1) L2 book bars
            fig_widget.add_bar(x=x_lvls, y=book_prev, row=1, col=1, marker_color='orange', name="Prev L2")
            fig_widget.add_bar(x=x_lvls, y=book_now,  row=1, col=1, marker_color=book_cols, name="Now L2")

            # (2) mid (absolute)
            fig_widget.add_scatter(x=np.arange(T), y=mid_series_abs, mode='lines',
                                   row=1, col=2, line=dict(width=1), name="Mid (abs)")
            fig_widget.add_shape(type="line", x0=t, x1=t,
                                 y0=float(np.nanmin(mid_series_abs)),
                                 y1=float(np.nanmax(mid_series_abs)),
                                 line=dict(color="green", width=2), xref="x2", yref="y2")

            # insertion markers on mid
            insert_dots_x = [p for p in valid_insertions if p < len(mid_series_abs)]
            insert_dots_y = [mid_series_abs[p] for p in insert_dots_x]
            fig_widget.add_scatter(
                x=insert_dots_x, y=insert_dots_y, mode='markers',
                marker=dict(size=7, symbol='circle'),
                text=insert_labels[:len(insert_dots_x)],
                hovertemplate="Insertion %{text}<extra></extra>",
                row=1, col=2, name="Insert marks"
            )

            # (3) raw log–log points & lines
            if xspan.size > 0:
                # allowed (impact>0)
                fig_widget.add_scatter(
                    x=log_qv[mask_pos], y=log_imp[mask_pos], mode='markers',
                    marker=dict(size=8),
                    text=[lbl for lbl, m in zip(insert_labels, mask_pos) if m],
                    hovertemplate="Insertion %{text}<extra></extra>",
                    row=1, col=3, name="Points (allowed)"
                )
                # zero-impact -> grey
                if np.any(~mask_pos):
                    fig_widget.add_scatter(
                        x=log_qv[~mask_pos], y=log_imp[~mask_pos], mode='markers',
                        marker=dict(size=8, color='grey'),
                        text=[lbl for lbl, m in zip(insert_labels, ~mask_pos) if m],
                        hovertemplate="Insertion %{text} (zero-impact)<extra></extra>",
                        row=1, col=3, name="Zero-impact (grey)"
                    )
                # fitted & theoretical lines
                fig_widget.add_scatter(x=xspan, y=fit_y, mode='lines',
                                       line=dict(dash='dot', width=2), row=1, col=3, name="Fit")
                fig_widget.add_scatter(x=xspan, y=th_y, mode='lines',
                                       line=dict(dash='dash', width=2), row=1, col=3, name=f"Theoretical (β={beta_theory:.2f})")

                fig_widget.update_xaxes(title="log(Q / V_exp)", row=1, col=3)
                fig_widget.update_yaxes(title="log(Impact)",   row=1, col=3)
            else:
                fig_widget.update_xaxes(title="log(Q / V_exp)", row=1, col=3)
                fig_widget.update_yaxes(title="log(Impact)",   row=1, col=3)

            # highlight current insertion
            if xspan.size > 0 and t in valid_insertions:
                i_sel = valid_insertions.index(t)
                x_sel = log_qv[i_sel]
                y_sel = log_imp[i_sel]
                fig_widget.add_scatter(
                    x=[x_sel], y=[y_sel], mode='markers',
                    marker=dict(color='red', size=12),
                    text=[insert_labels[i_sel]],
                    hovertemplate="Insertion %{text}<extra></extra>",
                    row=1, col=3, name="Current"
                )

        # raw message info @ t (display prices in dollars)
        m = messages[t].astype(int)
        price_dollars = m[PRICE_COL] / tick_size
        event_map = {1: "Limit", 2: "PartialCancel", 3: "Delete", 4: "Execution"}
        direction_map = {1: "Buy", 0: "Sell"}
        msg_box.value = (
            f"<b>{event_map.get(m[EVENT_TYPE_COL], '?')} • {direction_map.get(m[DIRECTION_COL], '?')} "
            f"• abs=${price_dollars:.2f} • rel={m[REL_COL]} • size={m[SIZE_COL]}</b><br>raw: {m.tolist()}"
        )

    def on_prev(_):
        if time_slider.value > time_slider.min:
            time_slider.value -= 1

    def on_next(_):
        if time_slider.value < time_slider.max:
            time_slider.value += 1

    id_dd.observe(lambda _: update_slider_range(), names='value')
    time_slider.observe(lambda _: update_plot(), names='value')
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)

    update_slider_range()
    update_plot()

    display(widgets.HBox([id_dd, btn_prev, btn_next, time_slider]))
    display(widgets.HBox([fig_widget, coeff_box]))
    display(msg_box)


In [18]:
all_betas = interactive_market_impact_plot(b_dict, m_dict, all_series, x, hist_steps, gen_block, num_insertions)

    'data': [],
    'layout': {'annotations': [{'font': {'size': 16},
          …

HTML(value='')

In [19]:
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import ipywidgets as widgets

def market_impact_dashboard_from_raw(
    b_seq_inp,            # dict[id] -> np.array(T, ...), where col 0 = Δmid per step (ticks)
    msg_seq_raw,          # dict[id] -> np.array(T, num_fields)
    all_series,           # unused for mid now
    x,                    # time axis (len T) - unused here
    hist_steps=550,
    gen_block=50,
    num_insertions=20,
    *,
    beta_theory=0.5,
    samples_used=None,
    special_first=(79, 15),
    show_fit=True,
    tick_size=100,        # convert ticks -> dollars before calling helper funcs
):
    """
    Same UI/figure as before, but x/y are computed via your helper functions:
      - impact/log_imp from calculate_impact(...)
      - V_exp/log_qv from calculate_market_volume(...), now fed with day-level execution_sum from sample_day_map
    α is fixed to ln(eta_day), where eta_day = ln(H/L)/0.8325546 using H,L from the same sample_day_map.
    β is the fixed-intercept slope: mean((y - α) / x).
    """

    # ------------- normalize inputs -------------
    if isinstance(b_seq_inp, pd.DataFrame):
        b_dict_local = {int(r.id): np.array(r.merged_data) for _, r in b_seq_inp.iterrows()}
    else:
        b_dict_local = b_seq_inp
    if isinstance(msg_seq_raw, pd.DataFrame):
        m_dict_local = {int(r.id): np.array(r.merged_data) for _, r in msg_seq_raw.iterrows()}
    else:
        m_dict_local = msg_seq_raw

    EVENT_TYPE_COL = 1
    PRICE_COL      = 3
    SIZE_COL       = 5

    eps = 1e-12
    tol = 1e-12

    # -------------------- compute x_df, y_df, coeffs_df via helpers --------------------
    def compute_tables():
        sample_ids = sorted(set(b_dict_local.keys()) & set(m_dict_local.keys()))
        col_names = [f"ins_{i}" for i in range(1, num_insertions + 1)]
        x_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
        y_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
        coeff_rows = []

        for sid in sample_ids:
            messages_ticks = m_dict_local[sid]
            book = b_dict_local[sid]
            T = len(messages_ticks)

            # insertion schedule
            insertion_positions = hist_steps + np.arange(1, num_insertions + 1) * gen_block
            valid_insertions = [pos for pos in insertion_positions if pos < T]
            if not valid_insertions:
                coeff_rows.append({"sample_id": sid, "alpha_hat": np.nan, "beta_hat": np.nan,
                                   "n_used": 0, "n_total": 0})
                continue

            # Reference price at first insertion (ticks -> $)
            ref_idx = valid_insertions[0]
            reference_price = float(messages_ticks[ref_idx, PRICE_COL]) / tick_size

            # -------- NEW: get day info from sample_day_map --------
            # Look up this sample_id in sample_day_map
            try:
                day_row = sample_day_map[sample_day_map['sample_id'] == sid]
                if not day_row.empty:
                    H_ticks = float(day_row.iloc[0]['highest_price'])
                    L_ticks = float(day_row.iloc[0]['lowest_price'])
                    execution_sum = float(day_row.iloc[0]['execution_sum'])
                else:
                    # Fallback if not found: infer H/L from pre-gen window and use window exec sum
                    H_ticks = float(np.max(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.max(messages_ticks[:, PRICE_COL]))
                    L_ticks = float(np.min(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.min(messages_ticks[:, PRICE_COL]))
                    exec_mask = (messages_ticks[:, EVENT_TYPE_COL].astype(int) == 4)
                    execution_sum = float(np.sum(messages_ticks[exec_mask, SIZE_COL].astype(float)))
            except Exception as _e:
                # Fallback if not found: infer H/L from pre-gen window and use window exec sum
                H_ticks = float(np.max(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.max(messages_ticks[:, PRICE_COL]))
                L_ticks = float(np.min(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.min(messages_ticks[:, PRICE_COL]))
                exec_mask = (messages_ticks[:, EVENT_TYPE_COL].astype(int) == 4)
                execution_sum = float(np.sum(messages_ticks[exec_mask, SIZE_COL].astype(float)))

            # Convert to dollars for Parkinson eta
            H = float(H_ticks) / tick_size
            L = float(L_ticks) / tick_size
            if np.isfinite(H) and np.isfinite(L) and H > L and L > 0:
                eta_day = np.log(H / L) / 0.8325546
                alpha_fixed = float(np.log(max(eta_day, eps)))   # α = ln(η)
            else:
                eta_day = eps
                alpha_fixed = float(np.log(eta_day))

            # Convert messages to dollars for the helper functions
            messages_dollars = messages_ticks.astype(float).copy()
            messages_dollars[:, PRICE_COL] /= tick_size

            # --- use your helper functions (unchanged graphs) ---
            impact, vwap_series, Q_cum, log_imp = calculate_impact(
                messages_dollars, valid_insertions, reference_price
            )
            # -------- NEW: pass execution_sum from sample_day_map into V_exp calc --------
            V_exp, log_qv = calculate_market_volume(
                messages_dollars, hist_steps, valid_insertions, execution_sum
            )

            # fill tables per insertion (same)
            mask_zero = impact <= tol
            mask_pos  = ~mask_zero
            for j, _idx in enumerate(valid_insertions):
                col = f"ins_{j+1}"
                if mask_zero[j] or not np.isfinite(log_qv[j]) or not np.isfinite(log_imp[j]):
                    x_df.loc[sid, col] = "ZERO"
                    y_df.loc[sid, col] = "ZERO"
                else:
                    x_df.loc[sid, col] = float(log_qv[j])
                    y_df.loc[sid, col] = float(log_imp[j])

            # per-sample β with fixed intercept (same)
            used_x = log_qv[mask_pos]
            used_y = log_imp[mask_pos]
            n_used = int(used_x.size)
            n_total = int(len(valid_insertions))
            if n_used >= 2 and np.all(np.isfinite(used_x)) and np.all(np.isfinite(used_y)):
                valid_mask = (used_x != 0) & np.isfinite(used_x) & np.isfinite(used_y)
                if np.sum(valid_mask) >= 2:
                    beta_hat = float(np.mean((used_y[valid_mask] - alpha_fixed) / used_x[valid_mask]))
                else:
                    beta_hat = np.nan
            else:
                beta_hat = np.nan

            coeff_rows.append({
                "sample_id": sid,
                "alpha_hat": alpha_fixed,   # fixed ln(η_day) from sample_day_map
                "beta_hat": beta_hat,
                "n_used": n_used,
                "n_total": n_total,
            })

        coeffs_df = pd.DataFrame.from_records(coeff_rows).set_index("sample_id").sort_index()
        return x_df, y_df, coeffs_df

    x_df, y_df, coeffs_df = compute_tables()

    # -------------------- tidy points (unchanged) --------------------
    all_ids = list(x_df.index)
    ordered_ids = [sid for sid in special_first if sid in all_ids]
    ordered_ids += [sid for sid in sorted(all_ids) if sid not in ordered_ids]
    if samples_used is not None:
        ordered_ids = ordered_ids[:samples_used]

    rows = []
    for sid in ordered_ids:
        for j, col in enumerate(x_df.columns, start=1):
            xv = x_df.loc[sid, col]
            yv = y_df.loc[sid, col]
            if isinstance(xv, (int, float, np.floating)) and isinstance(yv, (int, float, np.floating)):
                if np.isfinite(xv) and np.isfinite(yv):
                    rows.append({"sample_id": sid, "insertion": j, "x": float(xv), "y": float(yv)})
    points_df = pd.DataFrame(rows)
    if points_df.empty:
        print("No numeric points to plot.")
        return None, pd.DataFrame(), pd.DataFrame(), {}

    data_max_ins = int(points_df["insertion"].max())
    max_insertions = int(min(num_insertions, data_max_ins))
    a_values = np.arange(1, max_insertions + 1)

    # -------------------- histogram data (unchanged) --------------------
    betas_clean = coeffs_df["beta_hat"].replace([np.inf, -np.inf], np.nan).dropna().astype(float)
    beta_mean = float(betas_clean.mean()) if not betas_clean.empty else np.nan

    # -------------------- figure (unchanged) --------------------
    fig = go.FigureWidget(make_subplots(
        rows=2, cols=2,
        specs=[[{"type": "xy"}, {"type": "xy"}],
               [None,          {"type": "xy"}]],
        column_widths=[0.68, 0.32],
        row_heights=[0.55, 0.45],
        horizontal_spacing=0.07,
        vertical_spacing=0.12,
        subplot_titles=("Scatter & Global Fit", "β(a) evolution", "Distribution of β̂ across samples")
    ))

    LEGEND_MAX = 15
    palette = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#8c564b",
               "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#ff7f0e"]
    GREY = "rgba(0,0,0,0.25)"

    trace_meta = []
    for i, sid in enumerate(ordered_ids):
        sub = points_df[points_df["sample_id"] == sid].sort_values("insertion")
        x_vals = sub["x"].to_numpy()
        y_vals = sub["y"].to_numpy()
        ins = sub["insertion"].to_numpy()
        base_color = palette[i % len(palette)] if i != 1 else "#d62728"
        tr = go.Scatter(
            x=x_vals, y=y_vals, mode="markers",
            name=f"sample {sid} ({len(sub)}/{len(sub)} pts)",
            legendgroup=str(sid), showlegend=(i < LEGEND_MAX),
            marker=dict(size=7, color=base_color),
            hovertemplate=(
                "sample=%{customdata[0]}<br>"
                "ins=%{customdata[1]}<br>"
                "log(Q/V)=%{x:.4f}<br>"
                "log(Impact)=%{y:.4f}<extra></extra>"
            ),
            customdata=np.stack([sub["sample_id"].to_numpy(), ins], axis=1),
        )
        fig.add_trace(tr, row=1, col=1)
        trace_meta.append({"sid": sid, "ins": ins, "x": x_vals, "y": y_vals,
                           "base_color": base_color, "total": len(sub)})

    # global-fit trace (same)
    fit_trace_index = len(fig.data)
    fig.add_trace(
        go.Scatter(x=[], y=[], mode="lines", name="Global fit",
                   line=dict(dash="dash", width=2)),
        row=1, col=1
    )

    # ----- fitting helpers (unchanged, uses α_global = mean ln(η)) -----
    alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0

    def fit_for_mask(mask):
        X = points_df.loc[mask, "x"].to_numpy()
        Y = points_df.loc[mask, "y"].to_numpy()
        if len(Y) < 2:
            return np.nan, np.nan, np.nan, 0, np.array([]), np.array([])
        valid_mask = np.isfinite(X) & np.isfinite(Y) & (X != 0)
        if np.sum(valid_mask) < 2:
            return alpha_global, np.nan, np.nan, len(Y), np.array([]), np.array([])
        x_valid = X[valid_mask]; y_valid = Y[valid_mask]
        beta = float(np.mean((y_valid - alpha_global) / x_valid))
        y_pred = alpha_global + beta * X
        ss_tot = float(((Y - Y.mean()) ** 2).sum())
        ss_res = float(((Y - y_pred) ** 2).sum())
        r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
        x_min, x_max = X.min(), X.max()
        x_line = np.linspace(x_min, x_max, 200)
        y_line = alpha_global + beta * x_line
        return alpha_global, beta, r2, len(Y), x_line, y_line

    # precompute β(a) & fit lines (unchanged)
    a_values = np.arange(1, max_insertions + 1)
    betas_evo = np.full_like(a_values, np.nan, dtype=float)
    fit_lines = {}
    for idx, a in enumerate(a_values):
        mask = points_df["insertion"] >= a
        alpha, beta, r2, n, x_line, y_line = fit_for_mask(mask)
        betas_evo[idx] = beta
        fit_lines[a] = (x_line, y_line, alpha, beta, r2, n)

    # β(a) and indicator (unchanged)
    beta_line_idx = len(fig.data)
    fig.add_trace(
        go.Scatter(x=a_values, y=betas_evo, mode="lines+markers", name="β(a)"),
        row=1, col=2
    )
    beta_vline_idx = len(fig.data)
    y_min = float(np.nanmin(betas_evo)) if np.isfinite(betas_evo).any() else 0.0
    y_max = float(np.nanmax(betas_evo)) if np.isfinite(betas_evo).any() else 1.0
    fig.add_trace(
        go.Scatter(x=[a_values[0], a_values[0]], y=[y_min, y_max],
                   mode="lines", line=dict(color="green", dash="dot"), name="current a"),
        row=1, col=2
    )

    # histogram β̂ (unchanged)
    if not betas_clean.empty:
        fig.add_trace(go.Histogram(x=betas_clean.values, nbinsx=30, name="β̂"), row=2, col=2)
        fig.add_trace(go.Scatter(x=[beta_mean, beta_mean], y=[0, max(1, len(betas_clean))],
                                 mode="lines", name=f"Mean β̂ = {beta_mean:.4f}",
                                 line=dict(dash="dash")), row=2, col=2)
        fig.add_trace(go.Scatter(x=[beta_theory, beta_theory], y=[0, max(1, len(betas_clean))],
                                 mode="lines", name=f"β = {beta_theory:.2f} (theoretical)",
                                 line=dict(dash="dot")), row=2, col=2)

    # axes & layout (unchanged)
    fig.update_xaxes(title_text="log(Q / V_exp)", row=1, col=1)
    fig.update_yaxes(title_text="log(Impact)", row=1, col=1)
    fig.update_xaxes(title_text="a (insertion threshold)", row=1, col=2)
    fig.update_yaxes(title_text="β (slope)", row=1, col=2)
    fig.update_xaxes(title_text="β̂", row=2, col=2)
    fig.update_yaxes(title_text="Frequency", row=2, col=2)

    fig.update_layout(template="plotly_white", width=1500, height=800,
                      margin=dict(t=70, r=50, b=60, l=60),
                      legend=dict(orientation="v"))

    # fit box (unchanged)
    def set_fit_annotation(text_html: str):
        fig.layout.annotations = tuple(
            a for a in (fig.layout.annotations or [])
            if getattr(a, "name", "") != "fit_box"
        )
        fig.add_annotation(
            x=0.02, y=0.98, xref="paper", yref="paper",
            text=text_html, showarrow=False, align="left",
            bordercolor="lightgray", borderwidth=1, borderpad=8,
            bgcolor="rgba(245,245,245,1)", name="fit_box"
        )

    def update_left_fit(a_val: int):
        x_line, y_line, alpha, beta, r2, n = fit_lines.get(a_val, ([], [], np.nan, np.nan, np.nan, 0))
        fig.data[fit_trace_index].x = x_line
        fig.data[fit_trace_index].y = y_line
        if show_fit and n > 0 and np.isfinite(beta):
            fit_html = (
                "<b>Market Impact Fit (Fixed Intercept)</b><br>"
                f"<b>a:</b> {a_val}<br>"
                f"<b>Model:</b> log(Impact) = <b>{alpha:.6f}</b> + <b>{beta:.6f}</b> · log(Q/V)<br>"
                f"<b>α (fixed):</b> {alpha:.6f} (mean ln(η) across samples)<br>"
                f"<b>β:</b> {beta:.6f}<br>"
                f"<b>R²:</b> {r2:.4f}<br>"
                f"<b>Used points:</b> {n}"
            )
        else:
            fit_html = f"<b>Market Impact Fit (Fixed Intercept)</b><br><b>a:</b> {a_val}<br>No active points."
        set_fit_annotation(fit_html)

    def recolor_and_refit(a_val: int):
        for t_idx, meta in enumerate(trace_meta):
            active_mask = meta["ins"] >= a_val
            colors = [meta["base_color"] if ok else GREY for ok in active_mask]
            fig.data[t_idx].marker.color = colors
            active_count = int(np.count_nonzero(active_mask))
            fig.data[t_idx].name = f"sample {meta['sid']} ({meta['total']}/{active_count} pts)"
        if show_fit:
            update_left_fit(a_val)

        y_min_local = float(np.nanmin(betas_evo)) if np.isfinite(betas_evo).any() else 0.0
        y_max_local = float(np.nanmax(betas_evo)) if np.isfinite(betas_evo).any() else 1.0
        fig.data[beta_vline_idx].x = [a_val, a_val]
        fig.data[beta_vline_idx].y = [y_min_local, y_max_local]

    # controls (unchanged)
    a_slider = widgets.IntSlider(value=1, min=1, max=max_insertions, step=1, description="a")
    prev_btn = widgets.Button(description="Prev", layout=widgets.Layout(width="80px"))
    next_btn = widgets.Button(description="Next", layout=widgets.Layout(width="80px"))

    def on_prev(_):
        if a_slider.value > a_slider.min:
            a_slider.value -= 1

    def on_next(_):
        if a_slider.value < a_slider.max:
            a_slider.value += 1

    def on_a_change(change):
        if change["name"] == "value":
            recolor_and_refit(change["new"])

    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    a_slider.observe(on_a_change, names="value")

    # initial render
    recolor_and_refit(a_slider.value)
    display(widgets.HBox([prev_btn, next_btn, a_slider]), fig)

    controls = {"a_slider": a_slider, "prev_btn": prev_btn, "next_btn": next_btn}
    return fig, points_df, coeffs_df, controls

In [20]:
# Call the market impact dashboard function
fig, points_df, coeffs_df, controls = market_impact_dashboard_from_raw(
    b_seq_inp=b_dict,
    msg_seq_raw=m_dict,
    all_series=all_series,
    x=x,
    hist_steps=hist_steps,
    gen_block=gen_block,
    num_insertions=num_insertions,
    beta_theory=0.5,
    samples_used=None,
    special_first=(79, 15),
    show_fit=True
)

FigureWidget({
    'data': [{'customdata': {'bdata': 'cgJyA3IEcgVyBnIHcghyCXIK', 'dtype': 'i1', 'shape': '9, 2'},
              'hovertemplate': ('sample=%{customdata[0]}<br>ins' ... 'mpact)=%{y:.4f}<extra></extra>'),
              'legendgroup': '114',
              'marker': {'color': [#1f77b4, #1f77b4, #1f77b4, #1f77b4, #1f77b4,
                                   #1f77b4, #1f77b4, #1f77b4, #1f77b4],
                         'size': 7},
              'mode': 'markers',
              'name': 'sample 114 (9/9 pts)',
              'showlegend': True,
              'type': 'scatter',
              'uid': '0f3986d4-9d83-4cd9-992d-2cf39a2c83ca',
              'x': {'bdata': ('F+QuMO35C8Bi8+HQOG36v76ZlzNb8/' ... 'Ridsm02L/8P3BzQhvTvyeGY1Pybs2/'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('XfQGJa19KMC8RVwBXCshwO4xdo8amC' ... '5RMoKpHsCFFysFcU0ewOEnPiRU9B3A'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'cu

In [21]:
def compute_tables(b_seq_inp, msg_seq_raw, hist_steps, gen_block, num_insertions, beta_theory):
    if isinstance(b_seq_inp, pd.DataFrame):
        b_dict_local = {int(r.id): np.array(r.merged_data) for _, r in b_seq_inp.iterrows()}
    else:
        b_dict_local = b_seq_inp
    if isinstance(msg_seq_raw, pd.DataFrame):
        m_dict_local = {int(r.id): np.array(r.merged_data) for _, r in msg_seq_raw.iterrows()}
    else:
        m_dict_local = msg_seq_raw

    EVENT_TYPE_COL = 1
    PRICE_COL      = 3
    SIZE_COL       = 5

    eps = 1e-12

    sample_ids = sorted(set(b_dict_local.keys()) & set(m_dict_local.keys()))
    col_names = [f"ins_{i}" for i in range(1, num_insertions + 1)]
    x_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
    y_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
    coeff_rows = []

    def per_sample(sample_id):
        messages = m_dict_local[sample_id]
        T = len(messages)

        insertion_positions = hist_steps + np.arange(1, num_insertions + 1) * gen_block
        valid_insertions = [pos for pos in insertion_positions if pos < T]
        if not valid_insertions:
            return dict(valid_insertions=[], x_norm=None, y_norm=None,
                        mask_pos=None, mu_x=np.nan, mu_y=np.nan, std_x=np.nan, std_y=np.nan)

        # --- Reference price at the first insertion (in the same units as VWAP) ---
        ref_idx = valid_insertions[0]
        reference_price = float(messages[ref_idx, PRICE_COL])  # ticks are fine since ratio cancels units

        # --- Per-insertion Q_cum and VWAP (same units as reference_price) ---
        insert_sizes  = messages[valid_insertions, SIZE_COL].astype(float)
        insert_prices = messages[valid_insertions, PRICE_COL].astype(float)
        Q_cum    = np.cumsum(insert_sizes)
        notional = np.cumsum(insert_sizes * insert_prices)
        vwap_series = notional / np.maximum(Q_cum, eps)

        # ========================= NEW: pull day-level fields from sample_day_map =========================
        # Expect columns: ['sample_id','highest_price','lowest_price','execution_sum'] (prices in ticks)
        row = sample_day_map[sample_day_map['sample_id'] == sample_id]
        if row.empty:
            # Hard fallback (should be rare): use total executions in the whole sequence
            evt_types  = messages[:, EVENT_TYPE_COL].astype(int)
            exec_sizes = np.where(evt_types == 4, messages[:, SIZE_COL].astype(float), 0.0)
            execution_sum_day = float(np.sum(exec_sizes))
        else:
            execution_sum_day = float(row.iloc[0]['execution_sum'])
            # We also read H/L for completeness (not used here, since your current logic normalizes y)
            _H_ticks = float(row.iloc[0]['highest_price'])
            _L_ticks = float(row.iloc[0]['lowest_price'])
        # ================================================================================================

        # ---------- x = ln(Q_cum / execution_sum_day) ----------
        denom = max(execution_sum_day, eps)
        rel_size = Q_cum / denom
        log_qv = np.log(np.maximum(rel_size, eps))

        # ---------- y = ln(|VWAP / reference_price|) ----------
        impact_ratio = np.abs(vwap_series / max(reference_price, eps))
        log_imp = np.log(np.maximum(impact_ratio, eps))

        # Keep points where both are finite
        mask_pos = np.isfinite(log_qv) & np.isfinite(log_imp)
        used_x_raw = log_qv[mask_pos]
        used_y_raw = log_imp[mask_pos]

        # Per-sample standardization (your original logic)
        if used_x_raw.size >= 2:
            mu_x = float(used_x_raw.mean());  mu_y = float(used_y_raw.mean())
            std_x = float(np.sqrt(max(float(used_x_raw.var(ddof=0)), eps)))
            std_y = float(np.sqrt(max(float(used_y_raw.var(ddof=0)), eps)))
            x_norm_all = (log_qv - mu_x) / std_x
            y_norm_all = (log_imp - mu_y) / std_y
        else:
            mu_x = mu_y = np.nan
            std_x = std_y = np.nan
            x_norm_all = np.full_like(log_qv, np.nan, dtype=float)
            y_norm_all = np.full_like(log_imp, np.nan, dtype=float)

        return dict(valid_insertions=valid_insertions,
                    x_norm=x_norm_all, y_norm=y_norm_all,
                    mask_pos=mask_pos,
                    mu_x=mu_x, mu_y=mu_y, std_x=std_x, std_y=std_y)

    for sid in sample_ids:
        res = per_sample(sid)
        valid_ins = res["valid_insertions"]
        x_norm    = res["x_norm"]
        y_norm    = res["y_norm"]
        mask_pos  = res["mask_pos"]

        for j, _idx in enumerate(valid_ins):
            col = f"ins_{j+1}"
            xv = x_norm[j]; yv = y_norm[j]
            x_df.loc[sid, col] = float(xv) if np.isfinite(xv) else np.nan
            y_df.loc[sid, col] = float(yv) if np.isfinite(yv) else np.nan

        used_x = (x_norm[mask_pos] if x_norm is not None and mask_pos is not None else np.array([]))
        used_y = (y_norm[mask_pos] if y_norm is not None and mask_pos is not None else np.array([]))

        if used_x.size >= 2 and np.isfinite(used_x).sum() >= 2 and np.isfinite(used_y).sum() >= 2:
            reg = LinearRegression().fit(used_x.reshape(-1, 1), used_y)
            beta_hat  = float(reg.coef_[0])
            alpha_hat = float(reg.intercept_)
            mu_x = res["mu_x"]; mu_y = res["mu_y"]
            std_x = res["std_x"]; std_y = res["std_y"]
            slope_theory_norm = float(beta_theory * (std_x / std_y)) if (np.isfinite(std_x) and np.isfinite(std_y) and std_y > 0) else np.nan
            n_used = int(np.isfinite(used_x).sum())
        else:
            alpha_hat = beta_hat = slope_theory_norm = np.nan
            mu_x = res.get("mu_x", np.nan); mu_y = res.get("mu_y", np.nan)
            std_x = res.get("std_x", np.nan); std_y = res.get("std_y", np.nan)
            n_used = int(np.isfinite(used_x).sum()) if used_x.size else 0

        coeff_rows.append({
            "sample_id": sid,
            "alpha_hat": alpha_hat,
            "beta_hat": beta_hat,
            "slope_theory_norm": slope_theory_norm,
            "mu_x": mu_x, "std_x": std_x,
            "mu_y": mu_y, "std_y": std_y,
            "n_used": n_used,
            "n_total": int(len(valid_ins)),
        })

    coeffs_df = pd.DataFrame.from_records(coeff_rows).set_index("sample_id").sort_index()
    return x_df, y_df, coeffs_df

In [22]:
fig, points_df, coeffs_df, controls = market_impact_dashboard_from_raw(
    b_dict, m_dict, all_series, x,
    hist_steps=hist_steps,
    gen_block=gen_block,
    num_insertions=num_insertions,
    beta_theory=0.5,
    samples_used=None,
    special_first=(79, 15),
    show_fit=True,
    # nbins_hist=30
)

FigureWidget({
    'data': [{'customdata': {'bdata': 'cgJyA3IEcgVyBnIHcghyCXIK', 'dtype': 'i1', 'shape': '9, 2'},
              'hovertemplate': ('sample=%{customdata[0]}<br>ins' ... 'mpact)=%{y:.4f}<extra></extra>'),
              'legendgroup': '114',
              'marker': {'color': [#1f77b4, #1f77b4, #1f77b4, #1f77b4, #1f77b4,
                                   #1f77b4, #1f77b4, #1f77b4, #1f77b4],
                         'size': 7},
              'mode': 'markers',
              'name': 'sample 114 (9/9 pts)',
              'showlegend': True,
              'type': 'scatter',
              'uid': '278b8680-768d-4d31-8d23-8f44c10f0fb4',
              'x': {'bdata': ('F+QuMO35C8Bi8+HQOG36v76ZlzNb8/' ... 'Ridsm02L/8P3BzQhvTvyeGY1Pybs2/'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('XfQGJa19KMC8RVwBXCshwO4xdo8amC' ... '5RMoKpHsCFFysFcU0ewOEnPiRU9B3A'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'cu

In [23]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

def market_impact_dashboard_from_raw(
    b_seq_inp,            # dict[id] -> np.array(T, ...), where col 0 = Δmid per step (ticks)
    msg_seq_raw,          # dict[id] -> np.array(T, num_fields)
    all_series,           # unused (kept for compatibility)
    x,                    # time axis (len T, unused here)
    hist_steps=550,
    gen_block=50,
    num_insertions=20,
    *,
    beta_theory=0.5,
    samples_used=None,
    special_first=(79, 15),
    show_fit=True,
    nbins_hist=30,
    tick_size=100   # convert price ticks -> actual dollars (adjust if needed for data scale)
):
    """
    Creates an interactive market impact dashboard with a fixed-intercept model for impact:
      - Uses daily High/Low prices and total volume (from sample_day_map) to set a fixed intercept α = ln(η_day).
      - Calculates x = log(Q / V_exp) and y = log(Impact) without per-sample normalization.
      - Slider 'a' filters out the first (a-1) insertions from each sample to analyze β(a) evolution.
    Returns the Plotly FigureWidget, the points DataFrame, coefficients DataFrame, and control widgets.
    """
    # Ensure inputs are in dictionary form
    if isinstance(b_seq_inp, pd.DataFrame):
        b_dict_local = {int(r.id): np.array(r.merged_data) for _, r in b_seq_inp.iterrows()}
    else:
        b_dict_local = b_seq_inp
    if isinstance(msg_seq_raw, pd.DataFrame):
        m_dict_local = {int(r.id): np.array(r.merged_data) for _, r in msg_seq_raw.iterrows()}
    else:
        m_dict_local = msg_seq_raw

    # Columns indices for clarity (assuming specific format of msg_seq_raw)
    EVENT_TYPE_COL = 1  # e.g., 4 indicates an execution event
    PRICE_COL      = 3
    SIZE_COL       = 5

    eps = 1e-12
    tol = 1e-12

    sample_ids = sorted(set(b_dict_local.keys()) & set(m_dict_local.keys()))
    col_names = [f"ins_{i}" for i in range(1, num_insertions + 1)]
    x_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
    y_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
    coeff_rows = []

    # Function to process a single sample and compute its x and y values for each insertion
    def process_sample(sid):
        messages = m_dict_local[sid]
        T = len(messages)
        # Determine planned insertion indices for this sample
        insertion_positions = hist_steps + np.arange(1, num_insertions + 1) * gen_block
        valid_insertions = [pos for pos in insertion_positions if pos < T]
        if not valid_insertions:
            return None  # no valid insertion points for this sample

        # Reference price at first insertion (in tick units, convert to dollars)
        ref_idx = valid_insertions[0]
        reference_price = float(messages[ref_idx, PRICE_COL]) / tick_size

        # Compute cumulative inserted volume (Q_cum) and VWAP for those insertions
        insert_sizes  = messages[valid_insertions, SIZE_COL].astype(float)   # volumes of inserted orders
        insert_prices = messages[valid_insertions, PRICE_COL].astype(float) / tick_size  # prices of inserted orders in $
        Q_cum = np.cumsum(insert_sizes)  # cumulative quantity inserted
        notional = np.cumsum(insert_sizes * insert_prices)  # cumulative $ value of inserted volume
        vwap_series = notional / np.maximum(Q_cum, eps)     # VWAP after each insertion (in $)
        impact = np.abs(vwap_series - reference_price)      # absolute impact in $ after each insertion

        # Determine expected volume up to each insertion point.
        # Use sample_day_map (daily total volume) if available, otherwise fallback to cumulative exec volume from data.
        try:
            # sample_day_map is expected to be a global DataFrame with columns: sample_id, highest_price, lowest_price, execution_sum
            day_row = sample_day_map[sample_day_map['sample_id'] == sid]
            if not day_row.empty:
                H_ticks = float(day_row.iloc[0]['highest_price'])
                L_ticks = float(day_row.iloc[0]['lowest_price'])
                execution_sum = float(day_row.iloc[0]['execution_sum'])
            else:
                raise KeyError("Sample ID not found in sample_day_map")
        except Exception:
            # Fallback: compute High, Low from the historical pre-insertion window and total executed volume from the message data
            H_ticks = float(np.max(messages[:hist_steps, PRICE_COL])) if hist_steps < T else float(np.max(messages[:, PRICE_COL]))
            L_ticks = float(np.min(messages[:hist_steps, PRICE_COL])) if hist_steps < T else float(np.min(messages[:, PRICE_COL]))
            exec_mask = (messages[:, EVENT_TYPE_COL].astype(int) == 4)  # event type 4 = execution
            execution_sum = float(np.sum(messages[exec_mask, SIZE_COL].astype(float)))
        # Convert H, L from ticks to dollars for volatility calculation
        H = H_ticks / tick_size if L_ticks > 0 else np.nan
        L = L_ticks / tick_size if L_ticks > 0 else np.nan

        # Calculate daily volatility-based intercept α = ln(η_day), where η_day = ln(H/L)/0.8325546 (Parkinson estimator)
        if np.isfinite(H) and np.isfinite(L) and H > 0 and L > 0 and H > L:
            eta_day = np.log(H / L) / 0.8325546
            alpha_fixed = float(np.log(max(eta_day, eps)))
        else:
            alpha_fixed = float(np.log(eps))  # if data is not valid, use a very small number

        # Calculate expected volume (V_exp) at each insertion. We use total execution_sum for the day as baseline.
        # (Assumption: each insertion's expected volume ~ total daily volume up to that point; here we use full day volume for ratio.)
        V_exp_series = np.full(len(valid_insertions), execution_sum, dtype=float)  # use total daily volume for all insertions

        # Compute log values for x and y
        rel_size = Q_cum / np.maximum(V_exp_series, eps)  # Q/V_exp ratio for each insertion
        log_qv = np.log(np.maximum(rel_size, eps))
        log_imp = np.log(np.maximum(impact, eps))

        # Identify points with zero impact (to be marked as "ZERO" and excluded from fits)
        mask_zero = (impact <= tol) | ~np.isfinite(log_imp) | ~np.isfinite(log_qv)
        mask_pos  = ~mask_zero

        # Fill x_df, y_df for this sample
        for j, idx in enumerate(valid_insertions, start=1):
            col = f"ins_{j}"
            if j <= len(log_qv):
                if mask_zero[j-1]:  # j-1 corresponds to the index in log_qv/log_imp arrays
                    x_df.loc[sid, col] = "ZERO"
                    y_df.loc[sid, col] = "ZERO"
                else:
                    x_df.loc[sid, col] = float(log_qv[j-1])
                    y_df.loc[sid, col] = float(log_imp[j-1])
            else:
                # If an insertion number exceeds available data points (should not happen with valid_insertions logic)
                x_df.loc[sid, col] = np.nan
                y_df.loc[sid, col] = np.nan

        # Compute the fixed-intercept slope β_hat for this sample using all valid points
        used_x = log_qv[mask_pos]
        used_y = log_imp[mask_pos]
        n_used = int(used_x.size)
        n_total = len(valid_insertions)
        if n_used >= 2 and np.all(np.isfinite(used_x)) and np.all(np.isfinite(used_y)):
            # Exclude any zero X values from calculation to avoid division by zero
            valid_mask = (used_x != 0) & np.isfinite(used_x) & np.isfinite(used_y)
            if np.sum(valid_mask) >= 2:
                beta_hat = float(np.mean((used_y[valid_mask] - alpha_fixed) / used_x[valid_mask]))
            else:
                beta_hat = np.nan
        else:
            beta_hat = np.nan

        # Store coefficients for this sample
        coeff_rows.append({
            "sample_id": sid,
            "alpha_hat": alpha_fixed,  # fixed intercept = ln(η_day)
            "beta_hat": beta_hat,
            "n_used": n_used,
            "n_total": n_total
        })
        return None  # end of process_sample

    # Process each sample
    for sid in sample_ids:
        process_sample(sid)
    coeffs_df = pd.DataFrame.from_records(coeff_rows).set_index("sample_id").sort_index()

    # Build a long-form DataFrame of all points for plotting (only finite numeric points)
    all_ids = list(x_df.index)
    ordered_ids = [sid for sid in special_first if sid in all_ids]  # prioritize special_first samples
    ordered_ids += [sid for sid in sorted(all_ids) if sid not in ordered_ids]
    if samples_used is not None:
        ordered_ids = ordered_ids[:samples_used]

    points_list = []
    for sid in ordered_ids:
        for j, col in enumerate(x_df.columns, start=1):
            xv = x_df.loc[sid, col]
            yv = y_df.loc[sid, col]
            if isinstance(xv, (int, float)) and isinstance(yv, (int, float)):
                if np.isfinite(xv) and np.isfinite(yv):
                    points_list.append({"sample_id": sid, "insertion": j, "x": float(xv), "y": float(yv)})
    points_df = pd.DataFrame(points_list)
    if points_df.empty:
        print("No numeric points to plot.")
        return None, pd.DataFrame(), pd.DataFrame(), {}

    # Determine the maximum insertion number present in data (may be less than num_insertions if some samples have fewer points)
    max_data_ins = int(points_df["insertion"].max())
    max_insertions = int(min(num_insertions, max_data_ins))
    a_values = np.arange(1, max_insertions + 1)

    # Pre-calculate global β(a) and per-sample mean β̂(a) for each threshold a
    alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0  # global fixed intercept (mean of all sample alphas)
    betas_global = np.full_like(a_values, np.nan, dtype=float)
    betas_mean_per_sample = np.full_like(a_values, np.nan, dtype=float)
    fit_lines = {}

    def global_fit_for_mask(mask):
        """Compute global fit (fixed intercept = alpha_global) for points satisfying mask."""
        X = points_df.loc[mask, "x"].to_numpy()
        Y = points_df.loc[mask, "y"].to_numpy()
        n = len(Y)
        if n < 2:
            return alpha_global, np.nan, np.nan, n, np.array([]), np.array([])
        # Exclude zero or non-finite X values
        valid_mask = (X != 0) & np.isfinite(X) & np.isfinite(Y)
        if np.sum(valid_mask) < 2:
            return alpha_global, np.nan, np.nan, n, np.array([]), np.array([])
        X_val = X[valid_mask]
        Y_val = Y[valid_mask]
        # Calculate slope with fixed intercept (alpha_global)
        beta = float(np.mean((Y_val - alpha_global) / X_val))
        # Compute line for plotting
        x_min, x_max = X_val.min(), X_val.max()
        x_line = np.linspace(x_min, x_max, 200)
        y_line = alpha_global + beta * x_line
        # Calculate R² for informational purposes
        y_pred = alpha_global + beta * X_val
        ss_tot = float(np.sum((Y_val - Y_val.mean())**2))
        ss_res = float(np.sum((Y_val - y_pred)**2))
        r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
        return alpha_global, beta, r2, n, x_line, y_line

    def per_sample_betas_for_a(a):
        """Compute each sample's slope (fixed its own α) using points with insertion >= a, then return the series of β values."""
        betas = []
        for sid in ordered_ids:
            # Filter this sample's points at or beyond insertion a
            sub = points_df[(points_df["sample_id"] == sid) & (points_df["insertion"] >= a)]
            if len(sub) >= 2:
                X = sub["x"].to_numpy()
                Y = sub["y"].to_numpy()
                # Use this sample's fixed intercept α (from coeffs_df)
                alpha_s = coeffs_df.loc[sid, "alpha_hat"] if sid in coeffs_df.index else 0.0
                # Exclude zero or non-finite X values
                valid = (X != 0) & np.isfinite(X) & np.isfinite(Y)
                if np.sum(valid) >= 2:
                    beta_s = float(np.mean((Y[valid] - alpha_s) / X[valid]))
                    betas.append(beta_s)
                else:
                    betas.append(np.nan)
            else:
                betas.append(np.nan)
        s = pd.Series(betas, index=ordered_ids)
        return s.dropna()

    # Calculate global β(a) and mean of per-sample β̂(a) for each threshold value
    for idx, a in enumerate(a_values):
        mask = points_df["insertion"] >= a
        alpha, beta, r2, n, x_line, y_line = global_fit_for_mask(mask)
        betas_global[idx] = beta
        fit_lines[a] = (x_line, y_line, alpha, beta, r2, n)
        betas_a = per_sample_betas_for_a(a)
        betas_mean_per_sample[idx] = float(betas_a.mean()) if not betas_a.empty else np.nan

    # -------------------- Create Plotly Figure with subplots --------------------
    fig = go.FigureWidget(make_subplots(
        rows=2, cols=2,
        specs=[[{"type": "xy"}, None], [{"type": "xy"}, {"type": "xy"}]],
        column_widths=[0.55, 0.45],
        row_heights=[0.52, 0.48],
        horizontal_spacing=0.08,
        vertical_spacing=0.12,
        subplot_titles=("β(a) evolution", "", "Scatter & Global Fit", "Histogram of per-sample β̂(a)")
    ))

    # Top-left: β(a) evolution (global and per-sample mean slopes vs threshold)
    evo_global_idx = len(fig.data)
    fig.add_trace(go.Scatter(
        x=a_values, y=betas_global,
        mode="lines+markers", marker=dict(size=6),
        name="Global β(a)"
    ), row=1, col=1)
    evo_mean_idx = len(fig.data)
    fig.add_trace(go.Scatter(
        x=a_values, y=betas_mean_per_sample,
        mode="lines+markers", marker=dict(size=6),
        name="Samples Mean β̂(a)"
    ), row=1, col=1)
    # Vertical line indicating current 'a' value
    evo_vline_idx = len(fig.data)
    # Determine vertical line extents (min/max of the β(a) lines for initial full range)
    y_min_evo = float(np.nanmin(np.concatenate([betas_global, betas_mean_per_sample]))) if np.isfinite(betas_global).any() else 0.0
    y_max_evo = float(np.nanmax(np.concatenate([betas_global, betas_mean_per_sample]))) if np.isfinite(betas_mean_per_sample).any() else 1.0
    fig.add_trace(go.Scatter(
        x=[a_values[0], a_values[0]], y=[y_min_evo, y_max_evo],
        mode="lines", line=dict(color="green", dash="dot"),
        name="current a"
    ), row=1, col=1)
    # Reference horizontal lines at β = 0.5 (theoretical slope) and β = 0 (baseline)
    fig.add_trace(go.Scatter(
        x=[a_values.min(), a_values.max()], y=[beta_theory, beta_theory],
        mode="lines", line=dict(color="green", dash="dash"),
        name=f"β = {beta_theory:.2f} (theoretical)"
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=[a_values.min(), a_values.max()], y=[0, 0],
        mode="lines", line=dict(color="black", width=1),
        name="β = 0"
    ), row=1, col=1)
    fig.update_xaxes(title_text="a (insertion threshold)", row=1, col=1)
    fig.update_yaxes(title_text="β (slope)", row=1, col=1)

    # Bottom-left: Scatter plot of log(Q/V_exp) vs log(Impact) for each sample (points will be filtered by 'a')
    LEGEND_MAX = 15
    palette = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#8c564b",
               "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#ff7f0e"]
    trace_meta = []
    samples_start_idx = len(fig.data)
    for i, sid in enumerate(ordered_ids):
        sub_points = points_df[points_df["sample_id"] == sid].sort_values("insertion")
        x_vals = sub_points["x"].to_numpy()
        y_vals = sub_points["y"].to_numpy()
        ins_vals = sub_points["insertion"].to_numpy()
        base_color = palette[i % len(palette)]
        trace = go.Scatter(
            x=x_vals, y=y_vals,
            mode="markers",
            name=f"sample {sid} ({len(sub_points)}/{len(sub_points)} pts)",
            legendgroup=str(sid),
            showlegend=(i < LEGEND_MAX),
            marker=dict(size=7, color=base_color),
            customdata=np.stack([sub_points["sample_id"].to_numpy(), ins_vals], axis=1),
            hovertemplate=(
                "sample=%{customdata[0]}<br>"
                "ins=%{customdata[1]}<br>"
                "log(Q/V_exp)=%{x:.4f}<br>"
                "log(Impact)=%{y:.4f}<extra></extra>"
            )
        )
        fig.add_trace(trace, row=2, col=1)
        trace_meta.append({
            "sid": sid,
            "ins": ins_vals,
            "x_all": x_vals,
            "y_all": y_vals,
            "base_color": base_color,
            "total": len(sub_points)
        })
    # Add global fit line (will be updated with current 'a' selection)
    fit_trace_idx = len(fig.data)
    fig.add_trace(go.Scatter(
        x=[], y=[],
        mode="lines", name="Global fit",
        line=dict(dash="dash", width=2)
    ), row=2, col=1)
    fig.update_xaxes(title_text="log(Q / V_exp)", row=2, col=1)
    fig.update_yaxes(title_text="log(Impact)", row=2, col=1)

    # Bottom-right: Histogram of per-sample β̂ at current threshold 'a'
    hist_trace_idx = len(fig.data)
    fig.add_trace(go.Histogram(x=[], nbinsx=nbins_hist, name="β̂(a)"), row=2, col=2)
    mean_line_idx = len(fig.data)
    fig.add_trace(go.Scatter(
        x=[], y=[], mode="lines", name="Mean β̂(a)",
        line=dict(dash="dash")
    ), row=2, col=2)
    theory_line_idx = len(fig.data)
    fig.add_trace(go.Scatter(
        x=[beta_theory, beta_theory], y=[0, 1],
        mode="lines", name=f"β = {beta_theory:.2f} (theoretical)",
        line=dict(dash="dot")
    ), row=2, col=2)
    fig.update_xaxes(title_text="β̂(a)", row=2, col=2)
    fig.update_yaxes(title_text="Frequency", row=2, col=2)

    fig.update_layout(
        template="plotly_white",
        width=1500, height=840,
        margin=dict(t=70, r=50, b=60, l=60),
        legend=dict(orientation="v")
    )

    # Annotation box for global fit details
    def set_fit_annotation(text_html):
        # Remove any existing annotation with name 'fit_box'
        fig.layout.annotations = tuple(
            ann for ann in (fig.layout.annotations or [])
            if getattr(ann, "name", "") != "fit_box"
        )
        fig.add_annotation(
            x=0.30, y=0.26, xref="paper", yref="paper",
            text=text_html, showarrow=False, align="left",
            bordercolor="lightgray", borderwidth=1, borderpad=8,
            bgcolor="rgba(245,245,245,1)", name="fit_box"
        )

    # Update the global fit line and annotation for a given threshold a_val
    def update_scatter_fit(a_val):
        x_line, y_line, alpha, beta, r2, n = fit_lines.get(a_val, ([], [], alpha_global, np.nan, np.nan, 0))
        fig.data[fit_trace_idx].x = x_line
        fig.data[fit_trace_idx].y = y_line
        if show_fit and n > 0 and np.isfinite(beta):
            fit_html = (
                "<b>Market Impact Fit (Fixed Intercept)</b><br>"
                f"<b>a:</b> {a_val}<br>"
                f"<b>Model:</b> log(Impact) = <b>{alpha:.4f}</b> + <b>{beta:.4f}</b> · log(Q/V_exp)<br>"
                f"<b>α (fixed):</b> {alpha:.4f} (mean ln(η) across samples)<br>"
                f"<b>β:</b> {beta:.4f}<br>"
                f"<b>R²:</b> {r2:.4f}<br>"
                f"<b>Used points:</b> {n}"
            )
        else:
            fit_html = (
                "<b>Market Impact Fit (Fixed Intercept)</b><br>"
                f"<b>a:</b> {a_val}<br>No active points."
            )
        set_fit_annotation(fit_html)

    # Update histogram data and mean line for a given threshold a_val
    def update_histogram(a_val):
        betas_a = per_sample_betas_for_a(a_val)
        beta_values = betas_a.values if not betas_a.empty else np.array([])
        fig.data[hist_trace_idx].x = beta_values
        if beta_values.size > 0:
            counts, _ = np.histogram(beta_values, bins=nbins_hist)
            y_top = int(max(counts.max(), 1))
            beta_mean_val = float(np.mean(beta_values))
            fig.data[mean_line_idx].x = [beta_mean_val, beta_mean_val]
            fig.data[mean_line_idx].y = [0, y_top]
            fig.data[mean_line_idx].name = f"Mean β̂(a) = {beta_mean_val:.4f}"
        else:
            # No data for this threshold
            fig.data[mean_line_idx].x = []
            fig.data[mean_line_idx].y = []
            fig.data[mean_line_idx].name = "Mean β̂(a)"
            y_top = 1
        # Update the theoretical β vertical line height to match histogram
        fig.data[theory_line_idx].x = [beta_theory, beta_theory]
        fig.data[theory_line_idx].y = [0, max(y_top, 1)]

    # Redraw scatter points for each sample given threshold a_val (filters out early insertions)
    def redraw_scatter_for_a(a_val):
        for t_idx, meta in enumerate(trace_meta):
            active_mask = meta["ins"] >= a_val
            fig.data[samples_start_idx + t_idx].x = meta["x_all"][active_mask]
            fig.data[samples_start_idx + t_idx].y = meta["y_all"][active_mask]
            active_count = int(np.count_nonzero(active_mask))
            fig.data[samples_start_idx + t_idx].name = f"sample {meta['sid']} ({meta['total']}/{active_count} pts)"

    # Refresh all components (scatter, fit line, vertical indicator, histogram) for current slider value
    def refresh_all(a_val):
        redraw_scatter_for_a(a_val)
        if show_fit:
            update_scatter_fit(a_val)
        # Move vertical line on β(a) plot to current a
        y_min_local = float(np.nanmin(np.concatenate([betas_global, betas_mean_per_sample]))) if (np.isfinite(betas_global).any() or np.isfinite(betas_mean_per_sample).any()) else 0.0
        y_max_local = float(np.nanmax(np.concatenate([betas_global, betas_mean_per_sample]))) if (np.isfinite(betas_global).any() or np.isfinite(betas_mean_per_sample).any()) else 1.0
        fig.data[evo_vline_idx].x = [a_val, a_val]
        fig.data[evo_vline_idx].y = [y_min_local, y_max_local]
        update_histogram(a_val)

    # Set up interactive controls (slider and buttons)
    a_slider = widgets.IntSlider(value=1, min=1, max=max_insertions, step=1, description="a")
    prev_btn = widgets.Button(description="Prev", layout=widgets.Layout(width="80px"))
    next_btn = widgets.Button(description="Next", layout=widgets.Layout(width="80px"))

    def on_prev(_):
        if a_slider.value > a_slider.min:
            a_slider.value -= 1
    def on_next(_):
        if a_slider.value < a_slider.max:
            a_slider.value += 1
    def on_a_change(change):
        if change["name"] == "value":
            refresh_all(change["new"])

    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    a_slider.observe(on_a_change, names="value")

    # Initial render (for a = 1)
    refresh_all(a_slider.value)
    display(widgets.HBox([prev_btn, next_btn, a_slider]), fig)

    controls = {"a_slider": a_slider, "prev_btn": prev_btn, "next_btn": next_btn}
    return fig, points_df, coeffs_df, controls

# Example usage (assuming b_dict, m_dict, all_series, x are defined and sample_day_map is loaded as a DataFrame):
fig, points_df, coeffs_df, controls = market_impact_dashboard_from_raw(
    b_dict, m_dict, all_series, x,
    hist_steps=hist_steps,
    gen_block=gen_block,
    num_insertions=num_insertions,
    beta_theory=0.5,
    samples_used=None,      # limit number of samples if desired
    special_first=(79, 15),
    show_fit=True,
    nbins_hist=30,
    tick_size=100
)

FigureWidget({
    'data': [{'marker': {'size': 6},
              'mode': 'lines+markers',
              'name': 'Global β(a)',
              'type': 'scatter',
              'uid': '1aded52d-32fa-469c-9053-c4e57787f42c',
              'x': {'bdata': 'AQIDBAUGBwgJCgsMDQ4PEBESExQ=', 'dtype': 'i1'},
              'xaxis': 'x',
              'y': {'bdata': ('YtTWH2Q4QMBi1NYfZDhAwP/2ZkYDy0' ... 'aNQE3AjPZpGVbaTsA6T4/z1W9QwA=='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'marker': {'size': 6},
              'mode': 'lines+markers',
              'name': 'Samples Mean β̂(a)',
              'type': 'scatter',
              'uid': '538c6d62-238c-4ffb-bc5c-b4e78540b19f',
              'x': {'bdata': 'AQIDBAUGBwgJCgsMDQ4PEBESExQ=', 'dtype': 'i1'},
              'xaxis': 'x',
              'y': {'bdata': ('i8QZj/77QMCLxBmP/vtAwNoyzFLLs0' ... '9fD07AoPSWMVW1T8AAAAAAAAD4fw=='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'line'

In [24]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression

def global_beta_plot_from_raw(
    b_seq_inp,
    msg_seq_raw,
    all_series,
    x,
    hist_steps=550,
    gen_block=50,
    num_insertions=20,
    *,
    beta_theory=0.5,
    samples_used=None,
    special_first=(79, 15),
    nbins_hist=30
):
    def compute_tables(b_seq_inp, msg_seq_raw, hist_steps, gen_block, num_insertions, beta_theory):
        if isinstance(b_seq_inp, pd.DataFrame):
            b_dict_local = {int(r.id): np.array(r.merged_data) for _, r in b_seq_inp.iterrows()}
        else:
            b_dict_local = b_seq_inp
        if isinstance(msg_seq_raw, pd.DataFrame):
            m_dict_local = {int(r.id): np.array(r.merged_data) for _, r in msg_seq_raw.iterrows()}
        else:
            m_dict_local = msg_seq_raw

        EVENT_TYPE_COL = 1
        PRICE_COL      = 3
        SIZE_COL       = 5

        eps = 1e-12
        tol = 1e-12

        sample_ids = sorted(set(b_dict_local.keys()) & set(m_dict_local.keys()))
        col_names = [f"ins_{i}" for i in range(1, num_insertions + 1)]
        x_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
        y_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)

        def per_sample(sample_id):
            messages = m_dict_local[sample_id]
            T = len(messages)
            insertion_positions = hist_steps + np.arange(1, num_insertions + 1) * gen_block
            valid_insertions = [pos for pos in insertion_positions if pos < T]
            if not valid_insertions:
                return dict(valid_insertions=[], x_norm=None, y_norm=None,
                            mask_zero=None, mask_pos=None,
                            mu_x=np.nan, mu_y=np.nan, std_x=np.nan, std_y=np.nan)

            ref_idx = valid_insertions[0]
            reference_price_ticks = float(messages[ref_idx, PRICE_COL])

            insert_sizes  = messages[valid_insertions, SIZE_COL].astype(float)
            insert_prices = messages[valid_insertions, PRICE_COL].astype(float)
            Q_cum    = np.cumsum(insert_sizes)
            notional = np.cumsum(insert_sizes * insert_prices)
            vwap_ticks_series = notional / np.maximum(Q_cum, eps)
            impact_ticks = np.abs(vwap_ticks_series - reference_price_ticks)

            evt_types    = messages[:, EVENT_TYPE_COL].astype(int)
            exec_sizes   = np.where(evt_types == 4, messages[:, SIZE_COL].astype(float), 0.0)
            cum_exec_vol = np.cumsum(exec_sizes)
            V_exp = np.array([float(cum_exec_vol[idx-1] if (idx-1) >= 0 else 0.0)
                              for idx in valid_insertions])

            rel_size   = Q_cum / np.maximum(V_exp, eps)
            log_qv     = np.log(np.maximum(rel_size, eps))
            log_imp    = np.log(np.maximum(impact_ticks, eps))
            mask_zero  = impact_ticks <= tol
            mask_pos   = ~mask_zero

            used_x_raw = log_qv[mask_pos]
            used_y_raw = log_imp[mask_pos]

            if used_x_raw.size >= 2:
                mu_x = float(used_x_raw.mean());  mu_y = float(used_y_raw.mean())
                std_x = float(np.sqrt(max(float(used_x_raw.var(ddof=0)), eps)))
                std_y = float(np.sqrt(max(float(used_y_raw.var(ddof=0)), eps)))
                x_norm_all = (log_qv - mu_x) / std_x
                y_norm_all = (log_imp - mu_y) / std_y
            else:
                mu_x = mu_y = np.nan
                std_x = std_y = np.nan
                x_norm_all = np.full_like(log_qv, np.nan, dtype=float)
                y_norm_all = np.full_like(log_imp, np.nan, dtype=float)

            return dict(valid_insertions=valid_insertions,
                        x_norm=x_norm_all, y_norm=y_norm_all,
                        mask_zero=mask_zero, mask_pos=mask_pos,
                        mu_x=mu_x, mu_y=mu_y, std_x=std_x, std_y=std_y)

        for sid in sample_ids:
            res = per_sample(sid)
            valid_ins = res["valid_insertions"]
            x_norm    = res["x_norm"]
            y_norm    = res["y_norm"]
            mask_zero = res["mask_zero"]
            mask_pos  = res["mask_pos"]

            for j, _idx in enumerate(valid_ins):
                col = f"ins_{j+1}"
                if mask_zero[j]:
                    x_df.loc[sid, col] = "ZERO"
                    y_df.loc[sid, col] = "ZERO"
                else:
                    xv = x_norm[j]; yv = y_norm[j]
                    x_df.loc[sid, col] = float(xv) if np.isfinite(xv) else np.nan
                    y_df.loc[sid, col] = float(yv) if np.isfinite(yv) else np.nan

        return x_df, y_df

    x_df, y_df = compute_tables(
        b_seq_inp, msg_seq_raw, hist_steps, gen_block, num_insertions, beta_theory
    )

    all_ids = list(x_df.index)
    ordered_ids = [sid for sid in special_first if sid in all_ids]
    ordered_ids += [sid for sid in sorted(all_ids) if sid not in ordered_ids]
    if samples_used is not None:
        ordered_ids = ordered_ids[:samples_used]

    rows = []
    for sid in ordered_ids:
        for j, col in enumerate(x_df.columns, start=1):
            xv = x_df.loc[sid, col]
            yv = y_df.loc[sid, col]
            if isinstance(xv, (int, float, np.floating)) and isinstance(yv, (int, float, np.floating)):
                if np.isfinite(xv) and np.isfinite(yv):
                    rows.append({"sample_id": sid, "insertion": j, "x": float(xv), "y": float(yv)})
    points_df = pd.DataFrame(rows)
    if points_df.empty:
        print("No numeric points to plot.")
        return None, pd.DataFrame(), {}

    data_max_ins = int(points_df["insertion"].max())
    max_insertions = int(min(num_insertions, data_max_ins))
    a_values = np.arange(1, max_insertions + 1)

    def global_fit_for_mask(mask):
        X = points_df.loc[mask, "x"].to_numpy().reshape(-1, 1)
        Y = points_df.loc[mask, "y"].to_numpy()
        if len(Y) < 2:
            return np.nan
        reg = LinearRegression().fit(X, Y)
        beta = float(reg.coef_[0])
        return beta

    betas_global = np.full_like(a_values, np.nan, dtype=float)
    for idx, a in enumerate(a_values):
        mask = points_df["insertion"] >= a
        beta = global_fit_for_mask(mask)
        betas_global[idx] = beta

    # Plot only the global beta evolution
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=a_values,
        y=betas_global,
        mode="lines+markers",
        marker=dict(size=6),
        name="Global β(a)"
    ))
    fig.add_trace(go.Scatter(
        x=[a_values.min(), a_values.max()],
        y=[0.5, 0.5],
        mode="lines",
        line=dict(color="green", dash="dash"),
        name="y = 0.5"
    ))
    fig.add_trace(go.Scatter(
        x=[a_values.min(), a_values.max()],
        y=[0, 0],
        mode="lines",
        line=dict(color="black", width=1),
        name="y = 0"
    ))
    fig.update_xaxes(title_text="a (insertion threshold)")
    fig.update_yaxes(title_text="β (slope)")
    fig.update_layout(
        title="Global β(a) Evolution",
        template="plotly_white",
        width=800,
        height=500,
        margin=dict(t=60, r=30, b=50, l=60),
        legend=dict(orientation="v")
    )
    return fig, points_df

# Usage:
fig, points_df = global_beta_plot_from_raw(
    b_dict, m_dict, all_series, x,
    hist_steps=hist_steps,
    gen_block=gen_block,
    num_insertions=num_insertions,
    beta_theory=0.5,
    samples_used=None,
    special_first=(79, 15),
    nbins_hist=30
)
fig.show()

In [25]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

DEBUG = True
MAX_IDS_TO_PRINT = 20

def _safe_col(points_df, preferred, fallbacks):
    if preferred in points_df.columns:
        return preferred
    for c in fallbacks:
        if c in points_df.columns:
            return c
    return None

def debug_points_df(points_df, tag=""):
    print(f"\n=== [DEBUG] points_df summary {tag} ===")
    if points_df is None:
        print("points_df is None")
        return

    print(f"shape: {points_df.shape}")
    print(f"columns: {list(points_df.columns)}")
    if len(points_df) == 0:
        print("points_df is EMPTY.")
        return

    # Show head/tail small
    with pd.option_context('display.max_columns', None, 'display.width', 120):
        print("head(3):")
        print(points_df.head(3))
        print("tail(3):")
        print(points_df.tail(3))

    # NA counts
    na_counts = points_df.isna().sum()
    print("NA counts per column:")
    print(na_counts.to_dict())

    # Determine likely columns
    col_x = _safe_col(points_df, "log_qv", ["x", "log_QV", "log_qv"])
    col_y = _safe_col(points_df, "log_impact", ["y", "logImpact", "log_impact"])
    col_i = _safe_col(points_df, "ins_idx", ["ins_idx", "insertion_idx", "idx"])

    # Basic stats for X/Y
    if col_x:
        xvals = pd.to_numeric(points_df[col_x], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if len(xvals):
            print(f"{col_x}: count={len(xvals)}, min={xvals.min():.6f}, max={xvals.max():.6f}, mean={xvals.mean():.6f}, std={xvals.std():.6f}")
        else:
            print(f"{col_x}: no valid numeric values")
    else:
        print("Could not find X column (log_qv / x / log_QV).")

    if col_y:
        yvals = pd.to_numeric(points_df[col_y], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if len(yvals):
            print(f"{col_y}: count={len(yvals)}, min={yvals.min():.6f}, max={yvals.max():.6f}, mean={yvals.mean():.6f}, std={yvals.std():.6f}")
        else:
            print(f"{col_y}: no valid numeric values")
    else:
        print("Could not find Y column (log_impact / y / logImpact).")

    # Insertion index diagnostics
    if col_i:
        ii = pd.to_numeric(points_df[col_i], errors="coerce").dropna().astype(int)
        if len(ii):
            print(f"{col_i}: min={ii.min()}, max={ii.max()}, unique_count={ii.nunique()}")
            # Check gaps
            expected = set(range(ii.min(), ii.max()+1))
            missing = sorted(list(expected.difference(set(ii.unique()))))
            if missing:
                print(f"{col_i}: missing indices in [{ii.min()}..{ii.max()}]: {missing[:30]}{' ...' if len(missing)>30 else ''}")
            else:
                print(f"{col_i}: contiguous in [{ii.min()}..{ii.max()}]")
        else:
            print(f"{col_i}: no valid indices")
    else:
        print("No insertion index column found (ins_idx / insertion_idx / idx).")

def debug_beta_trace(beta_trace, tag=""):
    print(f"\n=== [DEBUG] beta_trace summary {tag} ===")
    if beta_trace is None:
        print("beta_trace is None")
        return

    x = np.asarray(beta_trace.x)
    y = np.asarray(beta_trace.y)

    print(f"len(x)={len(x)}, len(y)={len(y)}")
    if len(x) == 0:
        print("Empty beta trace.")
        return

    # Monotonicity of x
    mono = np.all(np.diff(x) >= 0)
    print(f"x range: [{np.min(x)}, {np.max(x)}], monotonic_non_decreasing={mono}")

    # First/last values
    print(f"first point: a={x[0]}, beta={y[0] if len(y)>0 else None}")
    print(f" last point: a={x[-1]}, beta={y[-1] if len(y)>0 else None}")

    # NaN/inf checks
    n_nan = int(np.sum(~np.isfinite(y)))
    print(f"β finite count={len(y)-n_nan}, non-finite count={n_nan}")

def recompute_full_beta(points_df):
    """OLS slope β over ALL points in points_df (if columns present)."""
    if points_df is None or len(points_df) < 2:
        return np.nan

    col_x = _safe_col(points_df, "log_qv", ["x", "log_QV", "log_qv"])
    col_y = _safe_col(points_df, "log_impact", ["y", "logImpact", "log_impact"])
    if (col_x is None) or (col_y is None):
        return np.nan

    X = pd.to_numeric(points_df[col_x], errors="coerce").replace([np.inf, -np.inf], np.nan)
    Y = pd.to_numeric(points_df[col_y], errors="coerce").replace([np.inf, -np.inf], np.nan)
    mask = X.notna() & Y.notna()
    X = X[mask].to_numpy().reshape(-1, 1)
    Y = Y[mask].to_numpy()
    if len(X) < 2:
        return np.nan
    lr = LinearRegression().fit(X, Y)
    return float(lr.coef_[0])

# =================== YOUR ORIGINAL BLOCK + DEBUG PRINTS ===================

cutoffs = np.round(np.arange(0.0, 1.0, 0.2), 2)  # Includes 0.0 to 0.9
filtered_sample_numbers = {}

for vcut in cutoffs:
    x_filt, all_series_filt, merged_filt, hist_steps_filt, gen_block_filt = prepare_volatility_filtered_series(
        merged, hist_msgs, n_gen_msgs, midprice_step_size, volatility_cutoff=vcut)
    filtered_sample_numbers[vcut] = list(merged_filt['id'])
    print(f"Cutoff {vcut}: {len(filtered_sample_numbers[vcut])} samples")

fig = go.Figure()
cutoff_colors = {
    0.0: "blue", 0.1: "cyan", 0.2: "orange", 0.3: "magenta", 0.4: "red",
    0.5: "brown", 0.6: "purple", 0.7: "gray", 0.8: "green", 0.9: "black"
}

# Ensure 0.0 is plotted first
ordered_cutoffs = sorted(filtered_sample_numbers.keys(), key=lambda x: (x != 0.0, x))

for vcut in ordered_cutoffs:
    sample_ids = filtered_sample_numbers[vcut]

    if DEBUG:
        print(f"\n=== [DEBUG] cutoff {vcut} ===")
        print(f"sample_ids[{len(sample_ids)}]: {sample_ids[:MAX_IDS_TO_PRINT]}{' ...' if len(sample_ids)>MAX_IDS_TO_PRINT else ''}")

    # Dicts filtered to available keys
    b_dict_filt = {k: b_dict[k] for k in sample_ids if k in b_dict}
    m_dict_filt = {k: m_dict[k] for k in sample_ids if k in m_dict}

    if DEBUG:
        missing_b = set(sample_ids) - set(b_dict_filt.keys())
        missing_m = set(sample_ids) - set(m_dict_filt.keys())
        if missing_b:
            print(f"[WARN] missing {len(missing_b)} ids in b_dict: {sorted(list(missing_b))[:MAX_IDS_TO_PRINT]}{' ...' if len(missing_b)>MAX_IDS_TO_PRINT else ''}")
        if missing_m:
            print(f"[WARN] missing {len(missing_m)} ids in m_dict: {sorted(list(missing_m))[:MAX_IDS_TO_PRINT]}{' ...' if len(missing_m)>MAX_IDS_TO_PRINT else ''}")

    fig_cut, points_df = global_beta_plot_from_raw(
        b_dict_filt, m_dict_filt, all_series, x,
        hist_steps=hist_steps,
        gen_block=gen_block,
        num_insertions=num_insertions,
        beta_theory=0.5,
        samples_used=None,
        special_first=(79, 15),
        nbins_hist=30
    )

    # ---- EXTRA DEBUG: tail-window composition per a ----
    def _safe_num(v, n=6):
        try:
            return float(np.round(v, n))
        except Exception:
            return v

    if points_df is not None and len(points_df):
        # Name mapping for safety
        COL_INS = "insertion"
        COL_X   = "x"   # your points_df uses 'x'/'y'
        COL_Y   = "y"

        if not {COL_INS, COL_X, COL_Y}.issubset(points_df.columns):
            print(f"[ERROR] points_df missing cols for debug: need {COL_INS},{COL_X},{COL_Y}")
        else:
            ins_min, ins_max = int(points_df[COL_INS].min()), int(points_df[COL_INS].max())
            print(f"[A-DEBUG] insertion range in points_df: [{ins_min}..{ins_max}]")
            # choose a grid of a's to print
            a_grid = list(range(ins_min, ins_max + 1))
            # or: a_grid = [ins_min, 5, 10, 15, ins_max]

            # Precompute a full OLS slope for comparison
            from sklearn.linear_model import LinearRegression
            lr = LinearRegression()
            X_all = points_df[COL_X].to_numpy().reshape(-1,1)
            y_all = points_df[COL_Y].to_numpy()
            lr.fit(X_all, y_all)
            beta_full_all = float(lr.coef_[0])
            print(f"[A-DEBUG] β_full over ALL points = {_safe_num(beta_full_all)}")

            # Per-a stats
            for a in a_grid:
                sub = points_df[points_df[COL_INS] >= a]
                n = len(sub)
                if n >= 2:
                    X = sub[COL_X].to_numpy().reshape(-1,1)
                    y = sub[COL_Y].to_numpy()
                    lr.fit(X, y)
                    beta_a = float(lr.coef_[0])
                else:
                    beta_a = np.nan

                used_ids = sub["sample_id"].unique() if "sample_id" in sub.columns else []
                print(
                    f"[A-DEBUG] a={a:>2d} | used_points={n:<4d} | β(a)={_safe_num(beta_a)} | "
                    f"ins_range=[{int(sub[COL_INS].min()) if n else '–'}..{int(sub[COL_INS].max()) if n else '–'}] | "
                    f"samples_used={len(used_ids)}"
                )

                # Optional: show a tiny preview of which insertions are present
                if n:
                    cnt_by_ins = sub[COL_INS].value_counts().sort_index()
                    preview = ", ".join([f"{i}:{cnt_by_ins[i]}" for i in cnt_by_ins.index[:5]])
                    tail_preview = ", ".join([f"{i}:{cnt_by_ins[i]}" for i in cnt_by_ins.index[-5:]])
                    print(f"          counts by insertion (head): {preview}")
                    if len(cnt_by_ins) > 5:
                        print(f"          counts by insertion (tail): {tail_preview}")


    if DEBUG:
        debug_points_df(points_df, tag=f"(cutoff={vcut})")

    if fig_cut is not None and len(fig_cut.data) > 0:
        beta_trace = fig_cut.data[0]

        if DEBUG:
            debug_beta_trace(beta_trace, tag=f"(cutoff={vcut})")

            # Cross‑check: full OLS on all points vs β at smallest a (tail window includes all points)
            beta_full = recompute_full_beta(points_df)
            try:
                a_vals = np.asarray(beta_trace.x)
                betas  = np.asarray(beta_trace.y)
                a_min_idx = int(np.nanargmin(a_vals)) if len(a_vals) else None
                beta_at_min_a = betas[a_min_idx] if a_min_idx is not None else np.nan
            except Exception:
                a_min_idx, beta_at_min_a = None, np.nan

            print(f"[CHECK] β_full(all points) = {beta_full:.6f}")
            print(f"[CHECK] β at min(a) (should match full if min(a)=0 and uses all points) = {beta_at_min_a:.6f}")

            if np.isfinite(beta_full) and np.isfinite(beta_at_min_a):
                diff = abs(beta_full - beta_at_min_a)
                if diff > 1e-3:
                    print(f"[WARN] β mismatch full vs min(a): diff={diff:.6f} (check filtering/weights)")

        label = f"0.0 β(a)" if vcut == 0.0 else f"{vcut} β(a)"
        fig.add_trace(go.Scatter(
            x=beta_trace.x,
            y=beta_trace.y,
            mode="lines+markers",
            marker=dict(size=6),
            name=label,
            line=dict(color=cutoff_colors.get(vcut, None))
        ))
    else:
        if DEBUG:
            print(f"[WARN] fig_cut is None or has no data for cutoff {vcut}")

# Visual guides
a_values = fig.data[0].x if len(fig.data) > 0 else np.arange(1, num_insertions + 1)
fig.add_trace(go.Scatter(
    x=[min(a_values), max(a_values)],
    y=[0.5, 0.5],
    mode="lines",
    line=dict(color="green", dash="dash"),
    name="y = 0.5"
))
fig.add_trace(go.Scatter(
    x=[min(a_values), max(a_values)],
    y=[0, 0],
    mode="lines",
    line=dict(color="black", width=1),
    name="y = 0"
))
fig.update_xaxes(title_text="a: minimum insertion threshold (only insertions with index ≥ a are used in β fit)")
fig.update_yaxes(title_text="β (slope)")
fig.update_layout(
    title="Global β(a) Evolution for Different Volatility Cutoffs",
    template="plotly_white",
    width=800,
    height=500,
    margin=dict(t=60, r=30, b=50, l=60),
    legend=dict(orientation="v"),
)
fig.show()


Before filtering: 56 samples

After filtering: 56 samples

Cutoff 0.0: 56 samples
Before filtering: 56 samples

After filtering: 45 samples

Cutoff 0.2: 45 samples
Before filtering: 56 samples

After filtering: 34 samples

Cutoff 0.4: 34 samples
Before filtering: 56 samples

After filtering: 23 samples

Cutoff 0.6: 23 samples
Before filtering: 56 samples

After filtering: 12 samples

Cutoff 0.8: 12 samples

=== [DEBUG] cutoff 0.0 ===
sample_ids[56]: [114, 648, 1895, 2486, 2787, 3489, 4447, 4668, 6556, 6779, 8306, 8349, 8604, 8858, 10954, 11171, 11828, 14205, 14606, 16120] ...
[A-DEBUG] insertion range in points_df: [2..20]
[A-DEBUG] β_full over ALL points = -0.095537
[A-DEBUG] a= 2 | used_points=1013 | β(a)=-0.095537 | ins_range=[2..20] | samples_used=56
          counts by insertion (head): 2:46, 3:55, 4:56, 5:56, 6:56
          counts by insertion (tail): 16:52, 17:52, 18:52, 19:52, 20:52
[A-DEBUG] a= 3 | used_points=967  | β(a)=-0.101475 | ins_range=[3..20] | samples_used=56
       

In [26]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

# =========================
# DEBUG CONFIG
# =========================
DBG_ON                 = True     # master switch
DBG_SAMPLE_LEVEL       = True     # per-sample prints inside build_points_for_sample
DBG_SAMPLE_LIMIT       = 5        # how many samples to print in detail (aggregate builder)
DBG_SHOW_ROWS          = 5        # how many rows to print from per-sample DF
DBG_PREFIX_FIT_SUMMARY = True     # summary per 'a' (prefix window)
DBG_PREFIX_FIT_GRID    = None     # set to list like [0, 1, 5, 10, 15, 19] to print only those 'a'; None -> all
DBG_WARN_EPS_HITS      = True     # warn when we clamp by eps
EPS                    = 1e-12

def _sn(x, n=6):  # safe number print
    try:
        return float(np.round(x, n))
    except Exception:
        return x

# ==== helper: build per-insertion points for ONE sample (id) ====
def build_points_for_sample(messages, hist_steps, gen_block, num_insertions,
                            price_col=3, size_col=5, event_type_col=1,
                            eps=EPS, sid=None):
    """
    Returns a DataFrame with columns: ['ins_idx','log_qv','log_impact'] for a single sample.
    • Insertions at positions: hist_steps + (1..num_insertions)*gen_block
    • Q: cumulative sum of inserted sizes (only at insertion indices)
    • V_exp: cumulative executed market volume (event_type==4) from index 0 up to idx-1
    • Impact: |VWAP_inserted - reference_price| (reference at first insertion)
    """
    T = len(messages)
    positions  = hist_steps + (np.arange(1, num_insertions + 1) * gen_block)
    insert_pos = [int(p) for p in positions if p < T]

    if DBG_ON and DBG_SAMPLE_LEVEL and (sid is not None):
        print(f"\n[build_points_for_sample] sid={sid} | T={T} | planned_ins={list(positions[:10])}{'...' if len(positions)>10 else ''}")
        print(f"[build_points_for_sample] sid={sid} | valid insert_pos={insert_pos}")

    if len(insert_pos) == 0:
        if DBG_ON and DBG_SAMPLE_LEVEL and (sid is not None):
            print(f"[build_points_for_sample] sid={sid} | NO valid insertions -> empty DF")
        return pd.DataFrame(columns=["ins_idx", "log_qv", "log_impact"])

    # reference price at first insertion
    ref_price = float(messages[insert_pos[0], price_col])

    # cumulative executed market volume from start (index 0)
    evt_types = messages[:, event_type_col].astype(int)
    sizes_all = messages[:, size_col].astype(float)
    exec_sizes = np.where(evt_types == 4, sizes_all, 0.0)
    cum_exec = np.cumsum(exec_sizes)

    # per-insertion quantities
    dQ         = messages[insert_pos, size_col].astype(float)     # ΔQ_i
    ins_prices = messages[insert_pos, price_col].astype(float)
    Q_cum      = np.cumsum(dQ)                                    # Q_i
    notional_ticks_cum = np.cumsum(dQ * ins_prices)
    vwap_ticks         = notional_ticks_cum / np.maximum(Q_cum, eps)
    impact_ticks       = np.abs(vwap_ticks - ref_price)

    # V_exp at each insertion i = cum_exec[idx-1]
    V_exp = []
    for idx in insert_pos:
        V_i = cum_exec[idx - 1] if (idx - 1) >= 0 else 0.0
        V_exp.append(float(max(V_i, 0.0)))
    V_exp = np.array(V_exp, dtype=float)

    rel_size = Q_cum / np.maximum(V_exp, eps)                     # Q / V_exp

    # eps diagnostics
    n_Q_zero   = int(np.sum(Q_cum <= 0))
    n_V_zero   = int(np.sum(V_exp <= 0))
    n_imp_zero = int(np.sum(impact_ticks <= 0))
    if DBG_ON and DBG_SAMPLE_LEVEL and (sid is not None):
        print(f"[build_points_for_sample] sid={sid} | ref_price={_sn(ref_price)}")
        print(f"[build_points_for_sample] sid={sid} | dQ[:5]={dQ[:5]}, Q_cum[:5]={Q_cum[:5]}")
        print(f"[build_points_for_sample] sid={sid} | V_exp[:5]={V_exp[:5]}")
        print(f"[build_points_for_sample] sid={sid} | impact_ticks[:5]={impact_ticks[:5]}")
        if DBG_WARN_EPS_HITS and (n_Q_zero or n_V_zero or n_imp_zero):
            print(f"[WARN][sid={sid}] zeros -> Q<=0:{n_Q_zero}, V<=0:{n_V_zero}, Impact<=0:{n_imp_zero} (clamped by eps={eps})")

    # logs
    log_qv     = np.log(np.maximum(rel_size, eps))
    log_impact = np.log(np.maximum(impact_ticks, eps))

    # ins_idx is 0..(len(insert_pos)-1) for this sample
    ins_idx = np.arange(len(insert_pos), dtype=int)

    df = pd.DataFrame({"ins_idx": ins_idx, "log_qv": log_qv, "log_impact": log_impact})

    if DBG_ON and DBG_SAMPLE_LEVEL and (sid is not None):
        print(f"[build_points_for_sample] sid={sid} | df.shape={df.shape}")
        print(df.head(DBG_SHOW_ROWS))

    return df

# ==== helper: aggregate across many samples (ids) ====
def build_points_across_samples(m_dict, sample_ids, hist_steps, gen_block, num_insertions):
    dfs = []
    printed = 0
    if DBG_ON:
        print(f"\n[build_points_across_samples] assembling from {len(sample_ids)} sample_ids")

    for sid in sample_ids:
        if sid not in m_dict:
            if DBG_ON:
                print(f"[build_points_across_samples][MISS] sid={sid} not in m_dict")
            continue
        df = build_points_for_sample(m_dict[sid], hist_steps, gen_block, num_insertions, sid=sid)
        if len(df):
            dfs.append(df)
            if DBG_ON and printed < DBG_SAMPLE_LIMIT:
                print(f"[build_points_across_samples] sid={sid} contributed {df.shape[0]} rows")
                printed += 1
        else:
            if DBG_ON and printed < DBG_SAMPLE_LIMIT:
                print(f"[build_points_across_samples] sid={sid} -> empty df")
                printed += 1

    out = pd.concat(dfs, ignore_index=True) if len(dfs) else pd.DataFrame(columns=["ins_idx","log_qv","log_impact"])
    if DBG_ON:
        print(f"[build_points_across_samples] TOTAL rows={out.shape[0]} | cols={list(out.columns)}")
        if out.shape[0]:
            with pd.option_context('display.max_columns', None, 'display.width', 120):
                print(out.describe().T)
                print("head:")
                print(out.head(DBG_SHOW_ROWS))
                print("tail:")
                print(out.tail(DBG_SHOW_ROWS))
    return out

# ==== helper: compute prefix β(a) on [0:a] ====
def compute_prefix_betas(points_df, num_insertions):
    """
    Fit y = c + β x using all points with ins_idx <= a, for a=0..A.
    Returns a_values, betas (np.ndarrays).
    """
    if points_df.empty:
        if DBG_ON:
            print("[compute_prefix_betas] points_df is EMPTY")
        return np.array([]), np.array([])

    if "ins_idx" not in points_df.columns or "log_qv" not in points_df.columns or "log_impact" not in points_df.columns:
        raise ValueError("[compute_prefix_betas] points_df must have columns: ins_idx, log_qv, log_impact")

    max_idx = int(points_df["ins_idx"].max())
    A = int(min(num_insertions - 1, max_idx))
    if DBG_ON and DBG_PREFIX_FIT_SUMMARY:
        print(f"\n[compute_prefix_betas] A={A} (ins_idx range: 0..{max_idx}), num_insertions={num_insertions}")

    a_vals, betas = [], []
    lr = LinearRegression()

    # Full-fit β for consistency check
    X_all = points_df["log_qv"].to_numpy().reshape(-1, 1)
    y_all = points_df["log_impact"].to_numpy()
    lr.fit(X_all, y_all)
    beta_full = float(lr.coef_[0])
    if DBG_ON and DBG_PREFIX_FIT_SUMMARY:
        print(f"[compute_prefix_betas] β_full(all points)={_sn(beta_full)}")

    a_iter = range(0, A + 1) if DBG_PREFIX_FIT_GRID is None else DBG_PREFIX_FIT_GRID
    for a in a_iter:
        sub = points_df[points_df["ins_idx"] <= a]
        n = len(sub)
        if n >= 2:
            X = sub["log_qv"].to_numpy().reshape(-1, 1)
            y = sub["log_impact"].to_numpy()
            lr.fit(X, y)
            beta = float(lr.coef_[0])
        else:
            beta = np.nan

        a_vals.append(a)
        betas.append(beta)

        if DBG_ON and DBG_PREFIX_FIT_SUMMARY:
            # A few quick stats on the subset
            x_min = _sn(np.min(sub["log_qv"])) if n else "–"
            x_max = _sn(np.max(sub["log_qv"])) if n else "–"
            y_min = _sn(np.min(sub["log_impact"])) if n else "–"
            y_max = _sn(np.max(sub["log_impact"])) if n else "–"
            print(f"[compute_prefix_betas] a={a:>2d} | used_points={n:<5d} | β(a)={_sn(beta)} | "
                  f"x∈[{x_min},{x_max}] y∈[{y_min},{y_max}]")

    # Final consistency: at a=max_idx should equal β_full
    if DBG_ON and DBG_PREFIX_FIT_SUMMARY:
        try:
            a_idx = (a_vals.index(A)) if isinstance(a_vals, list) else int(np.where(np.array(a_vals)==A)[0][0])
            beta_at_A = betas[a_idx]
            diff = abs(beta_full - beta_at_A)
            print(f"[compute_prefix_betas][CHECK] β_at_a=max={_sn(beta_at_A)} vs β_full={_sn(beta_full)} | diff={_sn(diff)}")
        except Exception as e:
            print(f"[compute_prefix_betas][CHECK] could not compare to β_full: {e}")

    return np.array(a_vals), np.array(betas)

# ================== YOUR EXISTING BLOCK (kept) ==================

cutoffs = np.round(np.arange(0.0, 1.0, 0.2), 2)  # Includes 0.0 to 0.9
filtered_sample_numbers = {}

for vcut in cutoffs:
    x_filt, all_series_filt, merged_filt, hist_steps_filt, gen_block_filt = prepare_volatility_filtered_series(
        merged, hist_msgs, n_gen_msgs, midprice_step_size, volatility_cutoff=vcut)
    filtered_sample_numbers[vcut] = list(merged_filt['id'])
    print(f"Cutoff {vcut}: {len(filtered_sample_numbers[vcut])} samples")

fig = go.Figure()
cutoff_colors = {
    0.0: "blue", 0.1: "cyan", 0.2: "orange", 0.3: "magenta", 0.4: "red",
    0.5: "brown", 0.6: "purple", 0.7: "gray", 0.8: "green", 0.9: "black"
}

# Ensure 0.0 is plotted first
ordered_cutoffs = sorted(filtered_sample_numbers.keys(), key=lambda x: (x != 0.0, x))

# === NEW: compute PREFIX β(a) curves per cutoff ===
for vcut in ordered_cutoffs:
    sample_ids = filtered_sample_numbers[vcut]

    # Build per-insertion points across all samples for this cutoff
    points_df = build_points_across_samples(
        m_dict, sample_ids,
        hist_steps=hist_steps,
        gen_block=gen_block,
        num_insertions=num_insertions
    )

    if points_df.empty:
        if DBG_ON:
            print(f"[main] cutoff={vcut}: points_df is empty, skipping")
        continue

    # Compute β(a) on [0:a]
    a_vals, betas = compute_prefix_betas(points_df, num_insertions=num_insertions)
    if a_vals.size == 0:
        if DBG_ON:
            print(f"[main] cutoff={vcut}: no a_vals, skipping")
        continue

    label = f"0.0 β(a, prefix)" if vcut == 0.0 else f"{vcut} β(a, prefix)"
    fig.add_trace(go.Scatter(
        x=a_vals,
        y=betas,
        mode="lines+markers",
        marker=dict(size=6),
        name=label,
        line=dict(color=cutoff_colors.get(vcut, None))
    ))

# Visual guides
if len(fig.data) > 0:
    a_values = fig.data[0].x
else:
    a_values = np.arange(0, max(1, num_insertions))

fig.add_trace(go.Scatter(
    x=[int(np.min(a_values)), int(np.max(a_values))],
    y=[0.5, 0.5],
    mode="lines",
    line=dict(color="green", dash="dash"),
    name="y = 0.5"
))
fig.add_trace(go.Scatter(
    x=[int(np.min(a_values)), int(np.max(a_values))],
    y=[0, 0],
    mode="lines",
    line=dict(color="black", width=1),
    name="y = 0"
))

fig.update_xaxes(title_text="a: maximum insertion index used in β fit (prefix window [0:a])")
fig.update_yaxes(title_text="β (slope)")
fig.update_layout(
    title="Global β(a) (PREFIX windows) for Different Volatility Cutoffs",
    template="plotly_white",
    width=800,
    height=500,
    margin=dict(t=60, r=30, b=50, l=60),
    legend=dict(orientation="v"),
)
fig.show()


Before filtering: 56 samples

After filtering: 56 samples

Cutoff 0.0: 56 samples
Before filtering: 56 samples

After filtering: 45 samples

Cutoff 0.2: 45 samples
Before filtering: 56 samples

After filtering: 34 samples

Cutoff 0.4: 34 samples
Before filtering: 56 samples

After filtering: 23 samples

Cutoff 0.6: 23 samples
Before filtering: 56 samples

After filtering: 12 samples

Cutoff 0.8: 12 samples

[build_points_across_samples] assembling from 56 sample_ids

[build_points_for_sample] sid=114 | T=1011 | planned_ins=[551, 602, 653, 704, 755, 806, 857, 908, 959, 1010]...
[build_points_for_sample] sid=114 | valid insert_pos=[551, 602, 653, 704, 755, 806, 857, 908, 959, 1010]
[build_points_for_sample] sid=114 | ref_price=904500.0
[build_points_for_sample] sid=114 | dQ[:5]=[ 110.    5.  613.  235. 1049.], Q_cum[:5]=[ 110.  115.  728.  963. 2012.]
[build_points_for_sample] sid=114 | V_exp[:5]=[ 508.  618.  663. 1319. 1604.]
[build_points_for_sample] sid=114 | impact_ticks[:5]=[  0.  

In [27]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

# ===================== CONFIG =====================
DEBUG = True
MAX_IDS_TO_PRINT = 20
EXACT_MIN_POINTS = 2   # min points required to fit β at a given 'a'

# ===================== DEBUG HELPERS =====================
def _safe_col(points_df, preferred, fallbacks):
    if preferred in points_df.columns:
        return preferred
    for c in fallbacks:
        if c in points_df.columns:
            return c
    return None

def debug_points_df(points_df, tag=""):
    print(f"\n=== [DEBUG] points_df summary {tag} ===")
    if points_df is None:
        print("points_df is None")
        return

    print(f"shape: {points_df.shape}")
    print(f"columns: {list(points_df.columns)}")
    if len(points_df) == 0:
        print("points_df is EMPTY.")
        return

    with pd.option_context('display.max_columns', None, 'display.width', 120):
        print("head(3):")
        print(points_df.head(3))
        print("tail(3):")
        print(points_df.tail(3))

    na_counts = points_df.isna().sum()
    print("NA counts per column:")
    print(na_counts.to_dict())

    col_x = _safe_col(points_df, "log_qv", ["x", "log_QV", "log_qv"])
    col_y = _safe_col(points_df, "log_impact", ["y", "logImpact", "log_impact"])
    col_i = _safe_col(points_df, "ins_idx", ["insertion", "ins_idx", "insertion_idx", "idx"])

    if col_x:
        xvals = pd.to_numeric(points_df[col_x], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if len(xvals):
            print(f"{col_x}: count={len(xvals)}, min={xvals.min():.6f}, max={xvals.max():.6f}, mean={xvals.mean():.6f}, std={xvals.std():.6f}")
        else:
            print(f"{col_x}: no valid numeric values")
    else:
        print("Could not find X column (log_qv / x / log_QV).")

    if col_y:
        yvals = pd.to_numeric(points_df[col_y], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if len(yvals):
            print(f"{col_y}: count={len(yvals)}, min={yvals.min():.6f}, max={yvals.max():.6f}, mean={yvals.mean():.6f}, std={yvals.std():.6f}")
        else:
            print(f"{col_y}: no valid numeric values")
    else:
        print("Could not find Y column (log_impact / y / logImpact).")

    if col_i:
        ii = pd.to_numeric(points_df[col_i], errors="coerce").dropna().astype(int)
        if len(ii):
            print(f"{col_i}: min={ii.min()}, max={ii.max()}, unique_count={ii.nunique()}")
            expected = set(range(ii.min(), ii.max()+1))
            missing = sorted(list(expected.difference(set(ii.unique()))))
            if missing:
                print(f"{col_i}: missing indices in [{ii.min()}..{ii.max()}]: {missing[:30]}{' ...' if len(missing)>30 else ''}")
            else:
                print(f"{col_i}: contiguous in [{ii.min()}..{ii.max()}]")
        else:
            print(f"{col_i}: no valid indices")
    else:
        print("No insertion index column found (ins_idx / insertion_idx / idx).")

def recompute_full_beta(points_df):
    """OLS slope β over ALL points in points_df (if columns present)."""
    if points_df is None or len(points_df) < 2:
        return np.nan

    col_x = _safe_col(points_df, "log_qv", ["x", "log_QV", "log_qv"])
    col_y = _safe_col(points_df, "log_impact", ["y", "logImpact", "log_impact"])
    if (col_x is None) or (col_y is None):
        return np.nan

    X = pd.to_numeric(points_df[col_x], errors="coerce").replace([np.inf, -np.inf], np.nan)
    Y = pd.to_numeric(points_df[col_y], errors="coerce").replace([np.inf, -np.inf], np.nan)
    mask = X.notna() & Y.notna()
    X = X[mask].to_numpy().reshape(-1, 1)
    Y = Y[mask].to_numpy()
    if len(X) < 2:
        return np.nan
    lr = LinearRegression().fit(X, Y)
    return float(lr.coef_[0])

# ===================== EXACT-a BETA =====================
def compute_exact_betas(points_df, num_insertions):
    """
    β_exact(a): fit y = c + β x using ONLY points with insertion == a.
    Returns arrays (a_values, betas_exact, counts_at_a).
    """
    if points_df is None or len(points_df) == 0:
        return np.array([]), np.array([]), np.array([])

    if "insertion" not in points_df.columns or "x" not in points_df.columns or "y" not in points_df.columns:
        print("[ERROR] points_df must contain columns: 'insertion','x','y'")
        return np.array([]), np.array([]), np.array([])

    a_min = int(points_df["insertion"].min())
    a_max = int(min(points_df["insertion"].max(), num_insertions))
    a_vals, betas, counts = [], [], []

    lr = LinearRegression()
    for a in range(a_min, a_max + 1):
        sub = points_df[points_df["insertion"] == a]
        n = len(sub)
        if n >= EXACT_MIN_POINTS:
            X = sub["x"].to_numpy().reshape(-1, 1)
            y = sub["y"].to_numpy()
            lr.fit(X, y)
            beta = float(lr.coef_[0])
        else:
            beta = np.nan
        a_vals.append(a)
        betas.append(beta)
        counts.append(n)
    return np.array(a_vals), np.array(betas), np.array(counts)

def debug_exact_betas(points_df, a_vals, betas, counts, tag=""):
    print(f"\n=== [A-DEBUG exact-a] {tag} ===")
    if points_df is not None and len(points_df):
        ins_min, ins_max = int(points_df["insertion"].min()), int(points_df["insertion"].max())
        print(f"[A-DEBUG] insertion range in points_df: [{ins_min}..{ins_max}]")

    beta_full = recompute_full_beta(points_df)
    print(f"[A-DEBUG] β_full over ALL points = {np.round(beta_full, 6)}")

    # Print row per a (compact)
    for a, b, n in zip(a_vals, betas, counts):
        sub = points_df[points_df["insertion"] == a]
        used_ids = sub["sample_id"].unique() if "sample_id" in sub.columns else []
        x_min = np.round(sub["x"].min(), 6) if n else None
        x_max = np.round(sub["x"].max(), 6) if n else None
        y_min = np.round(sub["y"].min(), 6) if n else None
        y_max = np.round(sub["y"].max(), 6) if n else None
        print(
            f"[A-DEBUG] a={a:>2d} | used_points={n:<4d} | β_exact(a)={np.round(b, 6)} | "
            f"x∈[{x_min},{x_max}] y∈[{y_min},{y_max}] | samples_used={len(used_ids)}"
        )

# ================== YOUR PIPELINE (unchanged pieces you rely on) ==================
# Assumes you already have:
# - prepare_volatility_filtered_series(...)
# - global_beta_plot_from_raw(...)  -> we only use its points_df output
# - dicts: b_dict, m_dict
# - arrays/ints: all_series, x, hist_steps, gen_block, num_insertions

cutoffs = np.round(np.arange(0.0, 1.0, 0.2), 2)  # 0.0 .. 0.8 (since your prints show step=0.2)
filtered_sample_numbers = {}

for vcut in cutoffs:
    x_filt, all_series_filt, merged_filt, hist_steps_filt, gen_block_filt = prepare_volatility_filtered_series(
        merged, hist_msgs, n_gen_msgs, midprice_step_size, volatility_cutoff=vcut)
    filtered_sample_numbers[vcut] = list(merged_filt['id'])
    print(f"Cutoff {vcut}: {len(filtered_sample_numbers[vcut])} samples")

fig = go.Figure()
cutoff_colors = {
    0.0: "blue", 0.1: "cyan", 0.2: "orange", 0.3: "magenta", 0.4: "red",
    0.5: "brown", 0.6: "purple", 0.7: "gray", 0.8: "green", 0.9: "black"
}

# Ensure 0.0 first
ordered_cutoffs = sorted(filtered_sample_numbers.keys(), key=lambda x: (x != 0.0, x))

for vcut in ordered_cutoffs:
    sample_ids = filtered_sample_numbers[vcut]

    if DEBUG:
        print(f"\n=== [DEBUG] cutoff {vcut} ===")
        print(f"sample_ids[{len(sample_ids)}]: {sample_ids[:MAX_IDS_TO_PRINT]}{' ...' if len(sample_ids)>MAX_IDS_TO_PRINT else ''}")

    # Filter dicts to available keys
    b_dict_filt = {k: b_dict[k] for k in sample_ids if k in b_dict}
    m_dict_filt = {k: m_dict[k] for k in sample_ids if k in m_dict}

    if DEBUG:
        missing_b = set(sample_ids) - set(b_dict_filt.keys())
        missing_m = set(sample_ids) - set(m_dict_filt.keys())
        if missing_b:
            print(f"[WARN] missing {len(missing_b)} ids in b_dict: {sorted(list(missing_b))[:MAX_IDS_TO_PRINT]}{' ...' if len(missing_b)>MAX_IDS_TO_PRINT else ''}")
        if missing_m:
            print(f"[WARN] missing {len(missing_m)} ids in m_dict: {sorted(list(missing_m))[:MAX_IDS_TO_PRINT]}{' ...' if len(missing_m)>MAX_IDS_TO_PRINT else ''}")

    # We call your function just to get the unified points_df (with columns: sample_id, insertion, x, y)
    fig_cut, points_df = global_beta_plot_from_raw(
        b_dict_filt, m_dict_filt, all_series, x,
        hist_steps=hist_steps,
        gen_block=gen_block,
        num_insertions=num_insertions,
        beta_theory=0.5,
        samples_used=None,
        special_first=(79, 15),
        nbins_hist=30
    )

    if DEBUG:
        debug_points_df(points_df, tag=f"(cutoff={vcut})")

    # ---- NEW: compute β at EXACT insertion a (not [a:]) ----
    a_vals, betas_exact, counts_at_a = compute_exact_betas(points_df, num_insertions=num_insertions)

    if a_vals.size == 0:
        print(f"[WARN] no exact-a betas for cutoff {vcut}")
        continue

    if DEBUG:
        debug_exact_betas(points_df, a_vals, betas_exact, counts_at_a, tag=f"(cutoff={vcut})")

    label = f"0.0 β(a, exact)" if vcut == 0.0 else f"{vcut} β(a, exact)"
    fig.add_trace(go.Scatter(
        x=a_vals,
        y=betas_exact,
        mode="lines+markers",
        marker=dict(size=6),
        name=label,
        line=dict(color=cutoff_colors.get(vcut, None))
    ))

# Visual guides
if len(fig.data) > 0:
    a_values = fig.data[0].x
else:
    a_values = np.arange(0, max(1, num_insertions))

fig.add_trace(go.Scatter(
    x=[int(np.min(a_values)), int(np.max(a_values))],
    y=[0.5, 0.5],
    mode="lines",
    line=dict(color="green", dash="dash"),
    name="y = 0.5"
))
fig.add_trace(go.Scatter(
    x=[int(np.min(a_values)), int(np.max(a_values))],
    y=[0, 0],
    mode="lines",
    line=dict(color="black", width=1),
    name="y = 0"
))

fig.update_xaxes(title_text="a: insertion index used in β fit (EXACT a only)")
fig.update_yaxes(title_text="β (slope)")
fig.update_layout(
    title="Global β(a) using EXACT-a windows (no [a:]) for Different Volatility Cutoffs",
    template="plotly_white",
    width=800,
    height=500,
    margin=dict(t=60, r=30, b=50, l=60),
    legend=dict(orientation="v"),
)
fig.show()


Before filtering: 56 samples

After filtering: 56 samples

Cutoff 0.0: 56 samples
Before filtering: 56 samples

After filtering: 45 samples

Cutoff 0.2: 45 samples
Before filtering: 56 samples

After filtering: 34 samples

Cutoff 0.4: 34 samples
Before filtering: 56 samples

After filtering: 23 samples

Cutoff 0.6: 23 samples
Before filtering: 56 samples

After filtering: 12 samples

Cutoff 0.8: 12 samples

=== [DEBUG] cutoff 0.0 ===
sample_ids[56]: [114, 648, 1895, 2486, 2787, 3489, 4447, 4668, 6556, 6779, 8306, 8349, 8604, 8858, 10954, 11171, 11828, 14205, 14606, 16120] ...

=== [DEBUG] points_df summary (cutoff=0.0) ===
shape: (1013, 4)
columns: ['sample_id', 'insertion', 'x', 'y']
head(3):
   sample_id  insertion         x         y
0        114          2 -2.694290 -2.746694
1        114          3  0.714929 -0.174592
2        114          4 -0.068871  0.027485
tail(3):
      sample_id  insertion         x         y
1010      38336         18 -0.619621  0.225048
1011      38336   

# Super debug

In [28]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

# ======= CONFIG =======
DEBUG = True
MAX_IDS_TO_PRINT = 30
MAX_VAL_PREVIEW = 10  # for value previews in debug

# Wider console prints
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 50)

# ======= HELPERS =======
def _safe_col(points_df, preferred, fallbacks):
    if preferred in points_df.columns:
        return preferred
    for c in fallbacks:
        if c in points_df.columns:
            return c
    return None

def _safe_num(v, n=6):
    try:
        return float(np.round(v, n))
    except Exception:
        return v

def _print_section(h):
    print("\n" + "="*24 + f" {h} " + "="*24)

def debug_points_df(points_df, tag=""):
    _print_section(f"[DEBUG] points_df summary {tag}")
    if points_df is None:
        print("points_df is None")
        return

    print(f"shape: {points_df.shape}")
    cols = list(points_df.columns)
    print(f"columns[{len(cols)}]: {cols}")
    if len(points_df) == 0:
        print("points_df is EMPTY.")
        return

    # Show head/tail
    with pd.option_context('display.max_columns', None, 'display.width', 200):
        print("head(5):")
        print(points_df.head(5))
        print("tail(5):")
        print(points_df.tail(5))

    # NA counts
    na_counts = points_df.isna().sum()
    print("NA counts per column (non-zero shown first):")
    nz_na = na_counts[na_counts > 0].sort_values(ascending=False)
    z_na = na_counts[na_counts == 0].sort_index()
    if len(nz_na):
        print(nz_na.to_dict())
    if len(z_na):
        print("zero-NA cols:", list(z_na.index))

    # Data types
    print("dtypes:")
    print(points_df.dtypes)

    # Likely columns
    col_x = _safe_col(points_df, "log_qv", ["x", "log_QV", "log_qv", "logQV"])
    col_y = _safe_col(points_df, "log_impact", ["y", "logImpact", "log_impact", "logImpactY"])
    col_i = _safe_col(points_df, "ins_idx", ["ins_idx", "insertion_idx", "idx", "insertion"])

    # Basic stats for X
    if col_x:
        xvals = pd.to_numeric(points_df[col_x], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if len(xvals):
            print(f"{col_x}: count={len(xvals)}, min={xvals.min():.6f}, q1={xvals.quantile(0.25):.6f}, "
                  f"median={xvals.median():.6f}, q3={xvals.quantile(0.75):.6f}, max={xvals.max():.6f}, "
                  f"mean={xvals.mean():.6f}, std={xvals.std():.6f}")
        else:
            print(f"{col_x}: no valid numeric values")
    else:
        print("Could not find X column (log_qv / x / log_QV / logQV).")

    # Basic stats for Y
    if col_y:
        yvals = pd.to_numeric(points_df[col_y], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if len(yvals):
            print(f"{col_y}: count={len(yvals)}, min={yvals.min():.6f}, q1={yvals.quantile(0.25):.6f}, "
                  f"median={yvals.median():.6f}, q3={yvals.quantile(0.75):.6f}, max={yvals.max():.6f}, "
                  f"mean={yvals.mean():.6f}, std={yvals.std():.6f}")
        else:
            print(f"{col_y}: no valid numeric values")
    else:
        print("Could not find Y column (log_impact / y / logImpact / log_impact).")

    # Insertion index diagnostics — one line for all insertions
    if col_i:
        ii_raw = pd.to_numeric(points_df[col_i], errors="coerce")
        ii = ii_raw.dropna().astype(int)
        if len(ii):
            print(f"{col_i}: min={ii.min()}, max={ii.max()}, unique_count={ii.nunique()}, total={len(ii)}")
            cnt_by_ins = ii.value_counts().sort_index()
            pairs = [f"{k}:{cnt_by_ins[k]}" for k in cnt_by_ins.index]
            print("insertion counts: " + ", ".join(pairs))
        else:
            print(f"{col_i}: no valid indices")
    else:
        print("No insertion index column found (ins_idx / insertion_idx / idx / insertion).")

    # Per-sample diagnostics (compact)
    if "sample_id" in points_df.columns:
        sid = points_df["sample_id"].dropna()
        print(f"sample_id: unique={sid.nunique()}, total_rows_with_id={sid.shape[0]}")


def debug_beta_trace(beta_trace, tag=""):
    _print_section(f"[DEBUG] beta_trace summary {tag}")
    if beta_trace is None:
        print("beta_trace is None")
        return

    x = np.asarray(beta_trace.x)
    y = np.asarray(beta_trace.y)
    print(f"len(x)={len(x)}, len(y)={len(y)}")
    if len(x) == 0:
        print("Empty beta trace.")
        return

    # Monotonicity of x
    mono = np.all(np.diff(x) >= 0)
    print(f"x range: [{np.min(x)}, {np.max(x)}], monotonic_non_decreasing={mono}")
    # Duplicates / gaps
    ux = np.unique(x)
    print(f"unique x count={len(ux)}; duplicates={'YES' if len(ux)<len(x) else 'NO'}")
    if len(ux) > 1:
        gaps = np.setdiff1d(np.arange(int(ux.min()), int(ux.max())+1), ux)
        if len(gaps):
            print(f"missing a-values in [{int(ux.min())}..{int(ux.max())}] (first 20): {gaps[:20]}")
        else:
            print("no missing a-values between min/max ✔")

    # First/last values
    print(f"first point: a={x[0]}, beta={y[0] if len(y)>0 else None}")
    print(f" last point: a={x[-1]}, beta={y[-1] if len(y)>0 else None}")

    # NaN/inf checks
    n_nonfinite = int(np.sum(~np.isfinite(y)))
    print(f"β finite count={len(y)-n_nonfinite}, non-finite count={n_nonfinite}")
    if n_nonfinite:
        bad_idx = np.where(~np.isfinite(y))[0]
        print(f"non-finite indices (first 20): {bad_idx[:20]}")

def recompute_full_beta(points_df):
    """OLS slope β over ALL points in points_df (if columns present)."""
    if points_df is None or len(points_df) < 2:
        return np.nan

    col_x = _safe_col(points_df, "log_qv", ["x", "log_QV", "log_qv", "logQV"])
    col_y = _safe_col(points_df, "log_impact", ["y", "logImpact", "log_impact", "logImpactY"])
    if (col_x is None) or (col_y is None):
        return np.nan

    X = pd.to_numeric(points_df[col_x], errors="coerce").replace([np.inf, -np.inf], np.nan)
    Y = pd.to_numeric(points_df[col_y], errors="coerce").replace([np.inf, -np.inf], np.nan)
    mask = X.notna() & Y.notna()
    X = X[mask].to_numpy().reshape(-1, 1)
    Y = Y[mask].to_numpy()
    if len(X) < 2:
        return np.nan
    lr = LinearRegression().fit(X, Y)
    return float(lr.coef_[0])

# =================== MAIN: **ONLY** CUTOFF = 0.0 ===================

cutoffs = [0.0]  # <-- only filtration 0.0
filtered_sample_numbers = {}

for vcut in cutoffs:
    x_filt, all_series_filt, merged_filt, hist_steps_filt, gen_block_filt = prepare_volatility_filtered_series(
        merged, hist_msgs, n_gen_msgs, midprice_step_size, volatility_cutoff=vcut
    )
    filtered_sample_numbers[vcut] = list(merged_filt['id'])
    print(f"Cutoff {vcut}: {len(filtered_sample_numbers[vcut])} samples")
    if DEBUG:
        print(f"Sample IDs (first {MAX_IDS_TO_PRINT}): {filtered_sample_numbers[vcut][:MAX_IDS_TO_PRINT]}{' ...' if len(filtered_sample_numbers[vcut])>MAX_IDS_TO_PRINT else ''}")

fig = go.Figure()
cutoff_colors = {0.0: "blue"}

# Only 0.0 plotted
for vcut in cutoffs:
    sample_ids = filtered_sample_numbers[vcut]

    if DEBUG:
        _print_section(f"[DEBUG] cutoff {vcut}")
        print(f"sample_ids[{len(sample_ids)}]: {sample_ids[:MAX_IDS_TO_PRINT]}{' ...' if len(sample_ids)>MAX_IDS_TO_PRINT else ''}")

    # Filter dicts to available keys
    b_dict_filt = {k: b_dict[k] for k in sample_ids if k in b_dict}
    m_dict_filt = {k: m_dict[k] for k in sample_ids if k in m_dict}

    if DEBUG:
        missing_b = set(sample_ids) - set(b_dict_filt.keys())
        missing_m = set(sample_ids) - set(m_dict_filt.keys())
        if missing_b:
            print(f"[WARN] missing {len(missing_b)} ids in b_dict (first {MAX_IDS_TO_PRINT}): {sorted(list(missing_b))[:MAX_IDS_TO_PRINT]}{' ...' if len(missing_b)>MAX_IDS_TO_PRINT else ''}")
        if missing_m:
            print(f"[WARN] missing {len(missing_m)} ids in m_dict (first {MAX_IDS_TO_PRINT}): {sorted(list(missing_m))[:MAX_IDS_TO_PRINT]}{' ...' if len(missing_m)>MAX_IDS_TO_PRINT else ''}")

    fig_cut, points_df = global_beta_plot_from_raw(
        b_dict_filt, m_dict_filt, all_series, x,
        hist_steps=hist_steps,
        gen_block=gen_block,
        num_insertions=num_insertions,
        beta_theory=0.5,
        samples_used=None,
        special_first=(79, 15),
        nbins_hist=30
    )

    # ===== EXTRA-DETAILED DEBUG =====
    if points_df is not None and len(points_df):
        # Column names expected by the plotting function's points_df
        # Commonly: ['sample_id','insertion','x','y', ...]
        COL_INS = "insertion" if "insertion" in points_df.columns else _safe_col(points_df, "insertion", ["ins_idx", "idx", "insertion_idx"])
        COL_X   = "x" if "x" in points_df.columns else _safe_col(points_df, "x", ["log_qv", "log_QV"])
        COL_Y   = "y" if "y" in points_df.columns else _safe_col(points_df, "y", ["log_impact", "logImpact"])

        if not {COL_INS, COL_X, COL_Y}.issubset(points_df.columns):
            print(f"[ERROR] points_df missing cols for deep debug: need {COL_INS},{COL_X},{COL_Y}")
        else:
            ins_min, ins_max = int(points_df[COL_INS].min()), int(points_df[COL_INS].max())
            print(f"[A-DEBUG] insertion range in points_df: [{ins_min}..{ins_max}] (count={points_df.shape[0]})")

            # Per-insertion composition & OLS on tail windows
            lr = LinearRegression()
            X_all = points_df[COL_X].to_numpy().reshape(-1,1)
            y_all = points_df[COL_Y].to_numpy()
            lr.fit(X_all, y_all)
            beta_full_all = float(lr.coef_[0])
            print(f"[A-DEBUG] β_full over ALL points = {_safe_num(beta_full_all)}")

            # Per-sample coverage preview at head/tail of insertion range
            if "sample_id" in points_df.columns:
                sid_by_ins = points_df.groupby(COL_INS)["sample_id"].nunique()
                print(f"[A-DEBUG] unique samples per insertion (first {MAX_VAL_PREVIEW}):")
                print(sid_by_ins.head(MAX_VAL_PREVIEW).to_dict())
                if len(sid_by_ins) > MAX_VAL_PREVIEW:
                    print(f"[A-DEBUG] unique samples per insertion (last  {MAX_VAL_PREVIEW}):")
                    print(sid_by_ins.tail(MAX_VAL_PREVIEW).to_dict())

            # Per‑a tail window β(a) and composition
            for a in range(ins_min, ins_max + 1):
                sub = points_df[points_df[COL_INS] >= a]
                n = len(sub)
                if n >= 2:
                    X = sub[COL_X].to_numpy().reshape(-1,1)
                    y = sub[COL_Y].to_numpy()
                    lr.fit(X, y)
                    beta_a = float(lr.coef_[0])
                else:
                    beta_a = np.nan

                used_ids = sub["sample_id"].unique() if "sample_id" in sub.columns else []
                ins_lo = int(sub[COL_INS].min()) if n else None
                ins_hi = int(sub[COL_INS].max()) if n else None

                print(f"[A-DEBUG] a={a:>2d} | used_points={n:<5d} | β(a)={_safe_num(beta_a)} | ins_range=[{ins_lo}..{ins_hi}] | samples_used={len(used_ids)}")
                if n:
                    cnt_by_ins = sub[COL_INS].value_counts().sort_index()
                    head_preview = ", ".join([f"{i}:{cnt_by_ins[i]}" for i in cnt_by_ins.index[:5]])
                    tail_preview = ", ".join([f"{i}:{cnt_by_ins[i]}" for i in cnt_by_ins.index[-5:]])
                    print(f"          counts by insertion (head): {head_preview}")
                    if len(cnt_by_ins) > 5:
                        print(f"          counts by insertion (tail): {tail_preview}")

    if DEBUG:
        debug_points_df(points_df, tag=f"(cutoff={vcut})")

    # ===== Trace diagnostics and cross-checks =====
    if fig_cut is not None and len(fig_cut.data) > 0:
        beta_trace = fig_cut.data[0]  # assumed first trace is β(a)
        if DEBUG:
            debug_beta_trace(beta_trace, tag=f"(cutoff={vcut})")

            # Cross‑check: full OLS on all points vs β at smallest a.
            beta_full = recompute_full_beta(points_df)
            try:
                a_vals = np.asarray(beta_trace.x, dtype=float)
                betas  = np.asarray(beta_trace.y, dtype=float)
                a_min_idx = int(np.nanargmin(a_vals)) if len(a_vals) else None
                beta_at_min_a = betas[a_min_idx] if a_min_idx is not None else np.nan
                a_min_val = a_vals[a_min_idx] if a_min_idx is not None else np.nan
            except Exception:
                a_min_idx, beta_at_min_a, a_min_val = None, np.nan, np.nan

            print(f"[CHECK] β_full(all points) = {_safe_num(beta_full)}")
            print(f"[CHECK] β at min(a)={_safe_num(a_min_val)} -> {_safe_num(beta_at_min_a)}")
            if np.isfinite(beta_full) and np.isfinite(beta_at_min_a):
                diff = abs(beta_full - beta_at_min_a)
                print(f"[CHECK] |β_full - β(min a)| = {diff:.8f}")
                if diff > 1e-3:
                    print(f"[WARN] β mismatch full vs min(a): diff={diff:.6f} (possible filtering/weights/window-definition difference)")

            # Additional check: replicate fig β(a) for a few points from points_df tail windows
            if points_df is not None and len(points_df):
                col_ins = "insertion" if "insertion" in points_df.columns else _safe_col(points_df, "insertion", ["ins_idx","idx","insertion_idx"])
                col_x = "x" if "x" in points_df.columns else _safe_col(points_df, "x", ["log_qv","log_QV"])
                col_y = "y" if "y" in points_df.columns else _safe_col(points_df, "y", ["log_impact","logImpact"])

                if {col_ins, col_x, col_y}.issubset(points_df.columns):
                    lr = LinearRegression()
                    for a_pick in [int(np.nanmin(a_vals)), int(np.nanmedian(a_vals)), int(np.nanmax(a_vals))]:
                        sub = points_df[points_df[col_ins] >= a_pick]
                        if len(sub) >= 2:
                            lr.fit(sub[col_x].to_numpy().reshape(-1,1), sub[col_y].to_numpy())
                            beta_manual = float(lr.coef_[0])
                        else:
                            beta_manual = np.nan

                        # lookup trace beta at a_pick (if exact match exists)
                        if np.isfinite(a_pick):
                            mask = (a_vals == a_pick)
                            beta_trace_val = float(betas[mask][0]) if np.any(mask) else np.nan
                        else:
                            beta_trace_val = np.nan

                        print(f"[XCHK] a={a_pick} -> manual β={_safe_num(beta_manual)} ; trace β={_safe_num(beta_trace_val)} ; diff={_safe_num(abs(beta_manual - beta_trace_val)) if (np.isfinite(beta_manual) and np.isfinite(beta_trace_val)) else 'nan'}")

        # Plot the single trace
        label = f"0.0 β(a)"
        fig.add_trace(go.Scatter(
            x=beta_trace.x,
            y=beta_trace.y,
            mode="lines+markers",
            marker=dict(size=6),
            name=label,
            line=dict(color=cutoff_colors.get(vcut, None))
        ))
    else:
        if DEBUG:
            print(f"[WARN] fig_cut is None or has no data for cutoff {vcut}")

# ===== Visual guides =====
a_values = fig.data[0].x if len(fig.data) > 0 else np.arange(1, num_insertions + 1)
fig.add_trace(go.Scatter(
    x=[min(a_values), max(a_values)],
    y=[0.5, 0.5],
    mode="lines",
    line=dict(color="green", dash="dash"),
    name="y = 0.5"
))
fig.add_trace(go.Scatter(
    x=[min(a_values), max(a_values)],
    y=[0, 0],
    mode="lines",
    line=dict(color="black", width=1),
    name="y = 0"
))
fig.update_xaxes(title_text="a: minimum insertion threshold (only insertions with index ≥ a are used in β fit)")
fig.update_yaxes(title_text="β (slope)")
fig.update_layout(
    title="Global β(a) Evolution — Volatility Cutoff = 0.0 (DETAILED DEBUG)",
    template="plotly_white",
    width=900,
    height=540,
    margin=dict(t=70, r=30, b=60, l=70),
    legend=dict(orientation="v"),
)
fig.show()


Before filtering: 56 samples

After filtering: 56 samples

Cutoff 0.0: 56 samples
Sample IDs (first 30): [114, 648, 1895, 2486, 2787, 3489, 4447, 4668, 6556, 6779, 8306, 8349, 8604, 8858, 10954, 11171, 11828, 14205, 14606, 16120, 16133, 16538, 16941, 18870, 19669, 20394, 20774, 21001, 21490, 22181] ...

======================== [DEBUG] cutoff 0.0 ========================
sample_ids[56]: [114, 648, 1895, 2486, 2787, 3489, 4447, 4668, 6556, 6779, 8306, 8349, 8604, 8858, 10954, 11171, 11828, 14205, 14606, 16120, 16133, 16538, 16941, 18870, 19669, 20394, 20774, 21001, 21490, 22181] ...
[A-DEBUG] insertion range in points_df: [2..20] (count=1013)
[A-DEBUG] β_full over ALL points = -0.095537
[A-DEBUG] unique samples per insertion (first 10):
{2: 46, 3: 55, 4: 56, 5: 56, 6: 56, 7: 56, 8: 56, 9: 56, 10: 56, 11: 52}
[A-DEBUG] unique samples per insertion (last  10):
{11: 52, 12: 52, 13: 52, 14: 52, 15: 52, 16: 52, 17: 52, 18: 52, 19: 52, 20: 52}
[A-DEBUG] a= 2 | used_points=1013  | β(a)=-0.0955

In [29]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

# ================= CONFIG =================
DEBUG = True
MAX_IDS_TO_PRINT = 30
START_PREFIX_AT = 1        # 1 → print [:1], [:2], ...; set 2 if you want to start from [:2]
PREFIX_MODE = "le"         # "lt" -> prefix [:a] means insertion < a  (strict; your current behavior)
                           # "le" -> prefix [:a] means insertion <= a (inclusive)

# Console formatting
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 50)

# ================ HELPERS ================
def _safe_col(points_df, preferred, fallbacks):
    if preferred in points_df.columns:
        return preferred
    for c in fallbacks:
        if c in points_df.columns:
            return c
    return None

def _safe_num(v, n=6):
    try:
        return float(np.round(v, n))
    except Exception:
        return v

def _print_section(h):
    print("\n" + "="*24 + f" {h} " + "="*24)

def _format_counts(mapping):
    # mapping: dict[int -> int] -> "{2: 80}, {3: 119}, ..."
    items = [f"{{{int(k)}: {int(mapping[k])}}}" for k in sorted(mapping.keys())]
    return ", ".join(items) + "."

def _format_pairs_inline(series_like):
    # series_like: pandas Series indexed by insertion -> count -> "2:80, 3:119, ..."
    pairs = [f"{int(k)}:{int(series_like.loc[k])}" for k in sorted(series_like.index)]
    return ", ".join(pairs) + "."

def debug_points_df(points_df, tag=""):
    _print_section(f"[DEBUG] points_df summary {tag}")
    if points_df is None:
        print("points_df is None")
        return

    print(f"shape: {points_df.shape}")
    cols = list(points_df.columns)
    print(f"columns[{len(cols)}]: {cols}")
    if len(points_df) == 0:
        print("points_df is EMPTY.")
        return

    with pd.option_context('display.max_columns', None, 'display.width', 200):
        print("head(5):")
        print(points_df.head(5))
        print("tail(5):")
        print(points_df.tail(5))

    # Likely columns
    col_i = _safe_col(points_df, "ins_idx", ["ins_idx", "insertion_idx", "idx", "insertion"])

    # Insertion index diagnostics — single line
    if col_i:
        ii_raw = pd.to_numeric(points_df[col_i], errors="coerce")
        ii = ii_raw.dropna().astype(int)
        if len(ii):
            print(f"insertion: min={ii.min()}, max={ii.max()}, unique_count={ii.nunique()}, total={len(ii)}")
            cnt_by_ins = ii.value_counts().sort_index()
            print("insertion counts: " + _format_pairs_inline(cnt_by_ins))
        else:
            print(f"{col_i}: no valid indices")
    else:
        print("No insertion index column found (ins_idx / insertion_idx / idx / insertion).")

    if "sample_id" in points_df.columns:
        sid = points_df["sample_id"].dropna()
        print(f"sample_id: unique={sid.nunique()}, total_rows_with_id={sid.shape[0]}")

def recompute_full_beta(points_df):
    """OLS slope β over ALL points in points_df (if columns present)."""
    if points_df is None or len(points_df) < 2:
        return np.nan
    col_x = _safe_col(points_df, "log_qv", ["x", "log_QV", "log_qv", "logQV"])
    col_y = _safe_col(points_df, "log_impact", ["y", "logImpact", "log_impact", "logImpactY"])
    if (col_x is None) or (col_y is None):
        return np.nan
    X = pd.to_numeric(points_df[col_x], errors="coerce").replace([np.inf, -np.inf], np.nan)
    Y = pd.to_numeric(points_df[col_y], errors="coerce").replace([np.inf, -np.inf], np.nan)
    mask = X.notna() & Y.notna()
    X = X[mask].to_numpy().reshape(-1, 1)
    Y = Y[mask].to_numpy()
    if len(X) < 2:
        return np.nan
    lr = LinearRegression().fit(X, Y)
    return float(lr.coef_[0])

def identity_checks(cnt_by_ins, counts_tail, counts_pref, ins_min, ins_max, total_points):
    _print_section("IDENTITY CHECKS")
    print("[A-DEBUG] points used per a for [a:]:")
    print(_format_counts(counts_tail))
    print("[A-DEBUG] points used per a for [:a]:")
    print(_format_counts(counts_pref))

    if PREFIX_MODE == "lt":
        mismatches = []
        for a in range(ins_min, ins_max + 1):
            t = counts_tail.get(a, 0)
            p = counts_pref.get(a, 0)
            if t + p != total_points or t != (total_points - p):
                mismatches.append((a, t, p, total_points - p, t + p))
        if mismatches:
            print("[CHECK] MISMATCHES (a, tail[a], prefix[a], total - prefix[a], tail[a]+prefix[a]):")
            for a, t, p, tot_minus_p, sum_tp in mismatches[:25]:
                print(f"  a={a}: tail={t}, prefix={p}, total-prefix={tot_minus_p}, sum={sum_tp}")
        else:
            print("[CHECK] For ALL a (strict): tail[a] + prefix[a] == total AND tail[a] == total - prefix[a] ✔")

        cnt_at_max = int(cnt_by_ins.loc[ins_max]) if ins_max in cnt_by_ins.index else 0
        tail_min = counts_tail.get(ins_min, 0)
        prefix_max = counts_pref.get(ins_max, 0)
        print(f"[DEMO] tail[{ins_min}] = {tail_min}; prefix[{ins_max}] = {prefix_max}; count@{ins_max} = {cnt_at_max}")
        print(f"[DEMO] tail[{ins_min}] - prefix[{ins_max}] = {tail_min - prefix_max}  (should equal count@{ins_max} = {cnt_at_max})")
        print("[DEMO] OK: tail[min] - prefix[max] == count@max ✔" if tail_min - prefix_max == cnt_at_max else "[DEMO] NOT OK")

    else:  # PREFIX_MODE == "le" (inclusive)
        mismatches = []
        for a in range(ins_min, ins_max + 1):
            t_next = counts_tail.get(a + 1, 0) if (a + 1) <= ins_max else 0
            p_inc  = counts_pref.get(a, 0)
            if t_next + p_inc != total_points or t_next != (total_points - p_inc):
                mismatches.append((a, t_next, p_inc, total_points - p_inc, t_next + p_inc))
        if mismatches:
            print("[CHECK] MISMATCHES (a, tail[a+1], prefix_inc[a], total - prefix_inc[a], tail[a+1]+prefix_inc[a]):")
            for a, t_next, p_inc, tot_minus_p, sum_tp in mismatches[:25]:
                print(f"  a={a}: tail[a+1]={t_next}, prefix_inc={p_inc}, total-prefix={tot_minus_p}, sum={sum_tp}")
        else:
            print("[CHECK] For ALL a (inclusive): tail[a+1] + prefix[a] == total AND tail[a+1] == total - prefix[a] ✔")

        # Correct DEMOs using the SAME 'a'
        for a_demo in [ins_min, (ins_min + ins_max)//2, ins_max]:
            t_next = counts_tail.get(a_demo + 1, 0) if (a_demo + 1) <= ins_max else 0
            p_inc  = counts_pref.get(a_demo, 0)
            diff   = t_next - (total_points - p_inc)
            print(f"[DEMO] a={a_demo}: tail[a+1]={t_next}, prefix[a]={p_inc}, total={total_points}")
            print(f"[DEMO] tail[a+1] - (total - prefix[a]) = {diff}  (should be 0)")
        # Also show [:max] equals total explicitly
        p_max = counts_pref.get(ins_max, 0)
        print(f"[DEMO] [:max] == total?  prefix[{ins_max}]={p_max}, total={total_points}, equal={p_max == total_points}")


# ================= MAIN (cutoff = 0.0) =================
cutoffs = [0.0]   # only filtration 0.0
filtered_sample_numbers = {}
fig = go.Figure()

# (You already have these in your env: merged, hist_msgs, n_gen_msgs, midprice_step_size,
#  b_dict, m_dict, all_series, x, hist_steps, gen_block, num_insertions,
#  and the helper global_beta_plot_from_raw)

for vcut in cutoffs:
    # Filter samples by volatility cutoff
    x_filt, all_series_filt, merged_filt, hist_steps_filt, gen_block_filt = prepare_volatility_filtered_series(
        merged, hist_msgs, n_gen_msgs, midprice_step_size, volatility_cutoff=vcut
    )

    filtered_sample_numbers[vcut] = list(merged_filt['id'])
    print(f"Cutoff {vcut}: {len(filtered_sample_numbers[vcut])} samples")
    if DEBUG:
        ids_preview = filtered_sample_numbers[vcut][:MAX_IDS_TO_PRINT]
        print(f"Sample IDs: {ids_preview}{' ...' if len(filtered_sample_numbers[vcut]) > MAX_IDS_TO_PRINT else ''}")

    # Filter dicts to available keys
    sample_ids = filtered_sample_numbers[vcut]
    b_dict_filt = {k: b_dict[k] for k in sample_ids if k in b_dict}
    m_dict_filt = {k: m_dict[k] for k in sample_ids if k in m_dict}

    # Build points_df using your existing helper (we only use points_df; we'll compute β ourselves)
    _fig_cut, points_df = global_beta_plot_from_raw(
        b_dict_filt, m_dict_filt, all_series, x,
        hist_steps=hist_steps,
        gen_block=gen_block,
        num_insertions=num_insertions,
        beta_theory=0.5,
        samples_used=None,
        special_first=(79, 15),
        nbins_hist=30
    )

    if DEBUG:
        debug_points_df(points_df, tag=f"(cutoff={vcut})")

    if points_df is None or len(points_df) == 0:
        print("[ERROR] No points_df returned; skipping β computations.")
        continue

    # Column selection
    COL_INS = "insertion" if "insertion" in points_df.columns else _safe_col(points_df, "insertion", ["ins_idx", "idx", "insertion_idx"])
    COL_X   = "x" if "x" in points_df.columns else _safe_col(points_df, "x", ["log_qv", "log_QV"])
    COL_Y   = "y" if "y" in points_df.columns else _safe_col(points_df, "y", ["log_impact", "logImpact"])
    if not {COL_INS, COL_X, COL_Y}.issubset(points_df.columns):
        print(f"[ERROR] points_df missing required cols: have {points_df.columns.tolist()}")
        continue

    # Insertion stats
    ins_series = pd.to_numeric(points_df[COL_INS], errors="coerce").dropna().astype(int)
    ins_min, ins_max = int(ins_series.min()), int(ins_series.max())
    cnt_by_ins = ins_series.value_counts().sort_index()
    total_points = int(ins_series.shape[0])

    # Build β and counts for windows
    lr = LinearRegression()

    # Tail [a:] windows
    a_tail = list(range(ins_min, ins_max + 1))
    betas_tail, counts_tail = [], {}
    for a in a_tail:
        sub = points_df[points_df[COL_INS] >= a]
        n = len(sub)
        counts_tail[a] = int(n)
        if n >= 2:
            lr.fit(sub[COL_X].to_numpy().reshape(-1, 1), sub[COL_Y].to_numpy())
            betas_tail.append(float(lr.coef_[0]))
        else:
            betas_tail.append(np.nan)

    # Prefix [:a] windows (strict or inclusive)
    a_pref = list(range(max(START_PREFIX_AT, 1), ins_max + 1))
    betas_pref, counts_pref = [], {}
    if PREFIX_MODE == "lt":
        for a in a_pref:
            sub = points_df[points_df[COL_INS] < a]
            n = len(sub)
            counts_pref[a] = int(n)
            if n >= 2:
                lr.fit(sub[COL_X].to_numpy().reshape(-1, 1), sub[COL_Y].to_numpy())
                betas_pref.append(float(lr.coef_[0]))
            else:
                betas_pref.append(np.nan)
    else:  # "le"
        for a in a_pref:
            sub = points_df[points_df[COL_INS] <= a]
            n = len(sub)
            counts_pref[a] = int(n)
            if n >= 2:
                lr.fit(sub[COL_X].to_numpy().reshape(-1, 1), sub[COL_Y].to_numpy())
                betas_pref.append(float(lr.coef_[0]))
            else:
                betas_pref.append(np.nan)

    # Identity checks to explain numbers (e.g., tail[2] vs prefix[20])
    identity_checks(cnt_by_ins, counts_tail, counts_pref, ins_min, ins_max, total_points)

    # Plot both β-curves
    fig.add_trace(go.Scatter(
        x=a_tail, y=betas_tail,
        mode="lines+markers",
        marker=dict(size=6),
        name="β(a) for [a:] (tail)"
    ))
    fig.add_trace(go.Scatter(
        x=a_pref, y=betas_pref,
        mode="lines+markers",
        marker=dict(size=6),
        name=f"β(a) for [:a] ({'strict < a' if PREFIX_MODE=='lt' else 'inclusive ≤ a'})"
    ))

# ===== Visual guides =====
if len(fig.data) > 0:
    all_x = np.concatenate([np.asarray(tr.x, dtype=float) for tr in fig.data if len(tr.x)])
    x_min, x_max = int(np.nanmin(all_x)), int(np.nanmax(all_x))
else:
    x_min, x_max = 1, 20

fig.add_trace(go.Scatter(
    x=[x_min, x_max],
    y=[0.5, 0.5],
    mode="lines",
    line=dict(dash="dash"),
    name="y = 0.5"
))
fig.add_trace(go.Scatter(
    x=[x_min, x_max],
    y=[0, 0],
    mode="lines",
    name="y = 0"
))

fig.update_xaxes(title_text="a (insertion index threshold)", dtick=1)
fig.update_yaxes(title_text="β (slope)")
fig.update_layout(
    title=f"Global β vs a — tail [a:] and prefix [:a] (Cutoff = 0.0, PREFIX_MODE='{PREFIX_MODE}')",
    template="plotly_white",
    width=980,
    height=560,
    margin=dict(t=70, r=30, b=60, l=70),
    legend=dict(orientation="v"),
)
fig.show()



Before filtering: 56 samples

After filtering: 56 samples

Cutoff 0.0: 56 samples
Sample IDs: [114, 648, 1895, 2486, 2787, 3489, 4447, 4668, 6556, 6779, 8306, 8349, 8604, 8858, 10954, 11171, 11828, 14205, 14606, 16120, 16133, 16538, 16941, 18870, 19669, 20394, 20774, 21001, 21490, 22181] ...

======================== [DEBUG] points_df summary (cutoff=0.0) ========================
shape: (1013, 4)
columns[4]: ['sample_id', 'insertion', 'x', 'y']
head(5):
   sample_id  insertion         x         y
0        114          2 -2.694290 -2.746694
1        114          3  0.714929 -0.174592
2        114          4 -0.068871  0.027485
3        114          5  0.970566  0.372904
4        114          6  0.408114  0.451961
tail(5):
      sample_id  insertion         x         y
1008      38336         16 -0.268575  0.356961
1009      38336         17 -0.245376  0.235637
1010      38336         18 -0.619621  0.225048
1011      38336         19 -0.486593  0.195606
1012      38336         20 -0.1737

In [30]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ================= CONFIG =================
DEBUG = True
MAX_IDS_TO_PRINT = 30
START_PREFIX_AT = 1
PREFIX_MODE = "le"         # "lt" -> insertion < a, "le" -> insertion <= a

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 50)

# ================ HELPERS ================
def _safe_col(points_df, preferred, fallbacks):
    if preferred in points_df.columns:
        return preferred
    for c in fallbacks:
        if c in points_df.columns:
            return c
    return None

def _safe_num(v, n=6):
    try:
        return float(np.round(v, n))
    except Exception:
        return v

def _print_section(h):
    print("\n" + "="*24 + f" {h} " + "="*24)

def _format_counts(mapping):
    items = [f"{{{int(k)}: {int(mapping[k])}}}" for k in sorted(mapping.keys())]
    return ", ".join(items) + "."

def _format_pairs_inline(series_like):
    pairs = [f"{int(k)}:{int(series_like.loc[k])}" for k in sorted(series_like.index)]
    return ", ".join(pairs) + "."

def debug_points_df(points_df, tag=""):
    _print_section(f"[DEBUG] points_df summary {tag}")
    if points_df is None:
        print("points_df is None")
        return
    print(f"shape: {points_df.shape}")
    cols = list(points_df.columns)
    print(f"columns[{len(cols)}]: {cols}")
    if len(points_df) == 0:
        print("points_df is EMPTY.")
        return
    with pd.option_context('display.max_columns', None, 'display.width', 200):
        print("head(5):")
        print(points_df.head(5))
        print("tail(5):")
        print(points_df.tail(5))
    col_i = _safe_col(points_df, "insertion", ["ins_idx", "insertion_idx", "idx"])
    if col_i:
        ii = pd.to_numeric(points_df[col_i], errors="coerce").dropna().astype(int)
        if len(ii):
            print(f"insertion: min={ii.min()}, max={ii.max()}, unique_count={ii.nunique()}, total={len(ii)}")
            cnt_by_ins = ii.value_counts().sort_index()
            print("insertion counts: " + _format_pairs_inline(cnt_by_ins))
    if "sample_id" in points_df.columns:
        sid = points_df["sample_id"].dropna()
        print(f"sample_id: unique={sid.nunique()}, total_rows_with_id={sid.shape[0]}")

def identity_checks(cnt_by_ins, counts_tail, counts_pref, ins_min, ins_max, total_points):
    _print_section("IDENTITY CHECKS")
    print("[A-DEBUG] points used per a for [a:]:")
    print(_format_counts(counts_tail))
    print("[A-DEBUG] points used per a for [:a]:")
    print(_format_counts(counts_pref))
    if PREFIX_MODE == "le":
        mismatches = []
        for a in range(ins_min, ins_max + 1):
            t_next = counts_tail.get(a + 1, 0) if (a + 1) <= ins_max else 0
            p_inc = counts_pref.get(a, 0)
            if t_next + p_inc != total_points:
                mismatches.append((a, t_next, p_inc))
        if mismatches:
            print("[CHECK] MISMATCHES:", mismatches)
        p_max = counts_pref.get(ins_max, 0)
        print(f"[DEMO] [:max] == total?  prefix[{ins_max}]={p_max}, total={total_points}, equal={p_max == total_points}")

# ================== CORE: replicate dashboard compute logic (no plotting) ==================
def _compute_points_and_coeffs_like_dashboard(
    b_dict_local, m_dict_local, *, hist_steps, gen_block, num_insertions, tick_size=100
):
    """
    Reproduces the x/y and per-sample alpha/beta from market_impact_dashboard_from_raw,
    including use of sample_day_map (H, L, execution_sum) and fixed-intercept per-sample beta.
    Returns points_df (sample_id, insertion, x, y) and coeffs_df (alpha_hat, beta_hat).
    """
    EVENT_TYPE_COL = 1
    PRICE_COL      = 3
    SIZE_COL       = 5
    eps = 1e-12
    tol = 1e-12

    sample_ids = sorted(set(b_dict_local.keys()) & set(m_dict_local.keys()))
    rows_points = []
    coeff_rows = []

    for sid in sample_ids:
        messages_ticks = m_dict_local[sid]
        book = b_dict_local[sid]  # not used, but kept for parity
        T = len(messages_ticks)

        # insertion schedule (same)
        insertion_positions = hist_steps + np.arange(1, num_insertions + 1) * gen_block
        valid_insertions = [pos for pos in insertion_positions if pos < T]
        if not valid_insertions:
            coeff_rows.append({"sample_id": sid, "alpha_hat": np.nan, "beta_hat": np.nan, "n_used": 0, "n_total": 0})
            continue

        # Reference price at first insertion (ticks -> $)
        ref_idx = valid_insertions[0]
        reference_price = float(messages_ticks[ref_idx, PRICE_COL]) / tick_size

        # ---- NEW: read H, L, execution_sum from sample_day_map (fallbacks identical to dashboard) ----
        try:
            day_row = sample_day_map[sample_day_map['sample_id'] == sid]
            if not day_row.empty:
                H_ticks = float(day_row.iloc[0]['highest_price'])
                L_ticks = float(day_row.iloc[0]['lowest_price'])
                execution_sum = float(day_row.iloc[0]['execution_sum'])
            else:
                H_ticks = float(np.max(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.max(messages_ticks[:, PRICE_COL]))
                L_ticks = float(np.min(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.min(messages_ticks[:, PRICE_COL]))
                exec_mask = (messages_ticks[:, EVENT_TYPE_COL].astype(int) == 4)
                execution_sum = float(np.sum(messages_ticks[exec_mask, SIZE_COL].astype(float)))
        except Exception:
            H_ticks = float(np.max(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.max(messages_ticks[:, PRICE_COL]))
            L_ticks = float(np.min(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.min(messages_ticks[:, PRICE_COL]))
            exec_mask = (messages_ticks[:, EVENT_TYPE_COL].astype(int) == 4)
            execution_sum = float(np.sum(messages_ticks[exec_mask, SIZE_COL].astype(float)))

        # Convert to dollars for Parkinson eta
        H = float(H_ticks) / tick_size
        L = float(L_ticks) / tick_size
        if np.isfinite(H) and np.isfinite(L) and H > L and L > 0:
            eta_day = np.log(H / L) / 0.8325546
            alpha_fixed = float(np.log(max(eta_day, eps)))   # α = ln(η)
        else:
            eta_day = eps
            alpha_fixed = float(np.log(eta_day))

        # Convert messages to dollars for helper functions
        messages_dollars = messages_ticks.astype(float).copy()
        messages_dollars[:, PRICE_COL] /= tick_size

        # ---- Use your helper functions exactly like in dashboard ----
        impact, vwap_series, Q_cum, log_imp = calculate_impact(
            messages_dollars, valid_insertions, reference_price
        )
        V_exp, log_qv = calculate_market_volume(
            messages_dollars, hist_steps, valid_insertions, execution_sum
        )

        # Collect per-insertion points
        mask_zero = impact <= tol
        mask_pos  = ~mask_zero
        used_x = []
        used_y = []
        for j, _idx in enumerate(valid_insertions):
            if mask_zero[j] or (not np.isfinite(log_qv[j])) or (not np.isfinite(log_imp[j])):
                continue
            rows_points.append({
                "sample_id": sid,
                "insertion": int(j + 1),
                "x": float(log_qv[j]),   # log(Q / V_exp) with V_exp from sample_day_map
                "y": float(log_imp[j])   # log(Impact)
            })
            used_x.append(log_qv[j])
            used_y.append(log_imp[j])

        used_x = np.asarray(used_x, dtype=float)
        used_y = np.asarray(used_y, dtype=float)
        n_used = int(np.sum(np.isfinite(used_x) & np.isfinite(used_y) & (used_x != 0)))
        n_total = int(len(valid_insertions))

        # per-sample β with fixed intercept (mean((y - α)/x))
        if n_used >= 2:
            valid_mask = np.isfinite(used_x) & np.isfinite(used_y) & (used_x != 0)
            if np.sum(valid_mask) >= 2:
                beta_hat = float(np.mean((used_y[valid_mask] - alpha_fixed) / used_x[valid_mask]))
            else:
                beta_hat = np.nan
        else:
            beta_hat = np.nan

        coeff_rows.append({
            "sample_id": sid,
            "alpha_hat": alpha_fixed,   # fixed ln(η_day)
            "beta_hat": beta_hat,
            "n_used": n_used,
            "n_total": n_total,
        })

    points_df = pd.DataFrame(rows_points)
    coeffs_df = pd.DataFrame.from_records(coeff_rows).set_index("sample_id").sort_index()
    return points_df, coeffs_df

# ================= MAIN (now uses new x/y and fixed-intercept slope) =================
cutoffs = [0.0, 0.2, 0.4, 0.6, 0.8]
filtered_sample_numbers = {}
fig = go.Figure()

for vcut in cutoffs:
    # Your existing filtering util (unchanged)
    x_filt, all_series_filt, merged_filt, hist_steps_filt, gen_block_filt = prepare_volatility_filtered_series(
        merged, hist_msgs, n_gen_msgs, midprice_step_size, volatility_cutoff=vcut
    )
    sample_ids = list(merged_filt['id'])
    filtered_sample_numbers[vcut] = sample_ids
    print(f"Cutoff {vcut}: {len(sample_ids)} samples")
    if DEBUG:
        ids_preview = sample_ids[:MAX_IDS_TO_PRINT]
        print(f"Sample IDs: {ids_preview}{' ...' if len(sample_ids) > MAX_IDS_TO_PRINT else ''}")

    # Filter your dicts
    b_dict_filt = {k: b_dict[k] for k in sample_ids if k in b_dict}
    m_dict_filt = {k: m_dict[k] for k in sample_ids if k in m_dict}

    # === NEW: compute points & coeffs with the exact same logic as the dashboard ===
    points_df, coeffs_df = _compute_points_and_coeffs_like_dashboard(
        b_dict_filt, m_dict_filt,
        hist_steps=hist_steps, gen_block=gen_block, num_insertions=num_insertions, tick_size=100
    )

    if DEBUG:
        debug_points_df(points_df, tag=f"(cutoff={vcut})")
    if points_df is None or len(points_df) == 0:
        continue

    # Column mapping
    COL_INS = "insertion" if "insertion" in points_df.columns else _safe_col(points_df, "insertion", ["ins_idx", "idx"])
    COL_X = "x" if "x" in points_df.columns else _safe_col(points_df, "x", ["log_qv"])
    COL_Y = "y" if "y" in points_df.columns else _safe_col(points_df, "y", ["log_impact"])

    ins_series = pd.to_numeric(points_df[COL_INS], errors="coerce").dropna().astype(int)
    ins_min, ins_max = int(ins_series.min()), int(ins_series.max())
    cnt_by_ins = ins_series.value_counts().sort_index()
    total_points = int(ins_series.shape[0])

    # === FIXED-INTERCEPT FIT: use α_global = mean(alpha_hat) exactly like the dashboard ===
    alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0

    def _fixed_intercept_beta(sub_df):
        """
        Compute beta with fixed intercept: beta = mean((y - alpha_global)/x)
        (matching your dashboard's fit_for_mask logic)
        """
        if sub_df is None or len(sub_df) < 2:
            return np.nan
        X = pd.to_numeric(sub_df[COL_X], errors="coerce").to_numpy(dtype=float)
        Y = pd.to_numeric(sub_df[COL_Y], errors="coerce").to_numpy(dtype=float)
        valid = np.isfinite(X) & np.isfinite(Y) & (X != 0)
        if np.sum(valid) < 2:
            return np.nan
        return float(np.mean((Y[valid] - alpha_global) / X[valid]))

    # Tail [a:]
    a_tail = list(range(ins_min, ins_max + 1))
    betas_tail, counts_tail = [], {}
    for a in a_tail:
        sub = points_df[points_df[COL_INS] >= a]
        counts_tail[a] = len(sub)
        betas_tail.append(_fixed_intercept_beta(sub))

    # Prefix [:a] (<= or < depending on PREFIX_MODE)
    a_pref = list(range(max(START_PREFIX_AT, 1), ins_max + 1))
    betas_pref, counts_pref = [], {}
    for a in a_pref:
        sub = points_df[points_df[COL_INS] <= a] if PREFIX_MODE == "le" else points_df[points_df[COL_INS] < a]
        counts_pref[a] = len(sub)
        betas_pref.append(_fixed_intercept_beta(sub))

    # Exact-at-a
    a_exact = list(range(ins_min, ins_max + 1))
    betas_exact, counts_exact = [], {}
    for a in a_exact:
        sub = points_df[points_df[COL_INS] == a]
        counts_exact[a] = len(sub)
        betas_exact.append(_fixed_intercept_beta(sub))

    identity_checks(cnt_by_ins, counts_tail, counts_pref, ins_min, ins_max, total_points)

    # Plot with legend labels including cutoff
    fig.add_trace(go.Scatter(
        x=a_tail, y=betas_tail, mode="lines+markers",
        name=f"Vol:{vcut} [a:] (tail)"
    ))
    fig.add_trace(go.Scatter(
        x=a_pref, y=betas_pref, mode="lines+markers",
        name=f"Vol:{vcut} [:a] (prefix)"
    ))
    fig.add_trace(go.Scatter(
        x=a_exact, y=betas_exact, mode="lines+markers",
        name=f"Vol:{vcut} [exact @ a]"
    ))

# Guides
if len(fig.data) > 0:
    all_x = np.concatenate([np.asarray(tr.x, dtype=float) for tr in fig.data if len(tr.x)])
    x_min, x_max = int(np.nanmin(all_x)), int(np.nanmax(all_x))
else:
    x_min, x_max = 1, 20
fig.add_trace(go.Scatter(x=[x_min, x_max], y=[0.5, 0.5], mode="lines", line=dict(dash="dash"), name="y = 0.5"))
fig.add_trace(go.Scatter(x=[x_min, x_max], y=[0, 0], mode="lines", name="y = 0"))

fig.update_xaxes(title_text="a (insertion index threshold)", dtick=1)
fig.update_yaxes(title_text="β (slope)")
fig.update_layout(
    title=f"Global β vs a — tail, prefix, exact (PREFIX_MODE='{PREFIX_MODE}')",
    template="plotly_white", width=980, height=560
)
fig.show()

Before filtering: 56 samples

After filtering: 56 samples

Cutoff 0.0: 56 samples
Sample IDs: [114, 648, 1895, 2486, 2787, 3489, 4447, 4668, 6556, 6779, 8306, 8349, 8604, 8858, 10954, 11171, 11828, 14205, 14606, 16120, 16133, 16538, 16941, 18870, 19669, 20394, 20774, 21001, 21490, 22181] ...

======================== [DEBUG] points_df summary (cutoff=0.0) ========================
shape: (1013, 4)
columns[4]: ['sample_id', 'insertion', 'x', 'y']
head(5):
   sample_id  insertion         x          y
0        114          2 -3.497034 -12.245462
1        114          3 -1.651666  -8.584686
2        114          4 -1.371913  -8.297078
3        114          5 -0.635082  -7.805457
4        114          6 -0.424744  -7.692938
tail(5):
      sample_id  insertion         x         y
1008      38336         16 -0.384725 -8.245250
1009      38336         17 -0.341292 -8.288683
1010      38336         18 -0.337501 -8.292474
1011      38336         19 -0.318759 -8.303014
1012      38336         20 -